# Evo-1 LIBERO — torchao W8A16 static weight-only quantization

Base flow: uploaded FP16 A100 baseline setup.  
Quant cells: adapted from the uploaded QuantVLA-style Q-cells, with the quant backend replaced by **torchao `Int8WeightOnlyConfig`** weight-only quantization.

Default scope:

```python
QUANT_SCOPE = "both"
```

Targets:
- LLM all Linear: attention `q/k/v/o` + MLP `gate/up/down` = expected 98.
- Action head: all `action_head.transformer_blocks.*` Linear modules found at runtime.
- Vision model, projector/mlp1, norms, embeddings, lm_head stay FP16/BF16.

Calibration:
- fixed LIBERO observation buffer
- FP16 teacher vs torchao W8A16 student drift stats
- context / ATM / OHB diagnostics
- ATM and OHB application are ON by default to preserve the QuantVLA-style mechanism; diagnostics are also collected.


In [1]:
# 0. GPU check
import torch, os, subprocess, sys, textwrap, json, re, time
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    raise RuntimeError("No GPU. Runtime > Change runtime type > GPU")


CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
BF16 supported: True


In [2]:
# 1. Mount Drive and define paths
from google.colab import drive
drive.mount("/content/drive")

REPO = "/content/drive/MyDrive/Evo-1"
EVO = f"{REPO}/Evo_1"
LIBERO_EVAL = f"{REPO}/LIBERO_evaluation"
CKPT_DIR = "/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO"
RESULTS = "/content/drive/MyDrive/Evo-1-results/fp16_resumable"

os.makedirs(RESULTS, exist_ok=True)
print(REPO, EVO, LIBERO_EVAL, CKPT_DIR, RESULTS, sep="\n")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Evo-1
/content/drive/MyDrive/Evo-1/Evo_1
/content/drive/MyDrive/Evo-1/LIBERO_evaluation
/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO
/content/drive/MyDrive/Evo-1-results/fp16_resumable


In [3]:
# 2. Clone fresh official Evo-1 or restore original files
%cd /content/drive/MyDrive
if not os.path.exists(REPO):
    !git clone https://github.com/MINT-SJTU/Evo-1.git Evo-1
else:
    print("Repo already exists:", REPO)

%cd "$REPO"
!git restore Evo_1/scripts/Evo1_server.py || true
!git restore LIBERO_evaluation/libero_client_4tasks.py || true
!git status --short


/content/drive/MyDrive
Repo already exists: /content/drive/MyDrive/Evo-1
/content/drive/MyDrive/Evo-1
 M .gitignore
 M Evo_1/dataset/config.yaml
 M Evo_1/ds_config.json
 M Evo_1/model/action_head/flow_matching.py
 M Evo_1/scripts/Evo1_server.py
 M MetaWorld_evaluation/mt50_evo1_client_prompt.py
 M MetaWorld_evaluation/tasks.jsonl
 M so100_evo1/lerobot-main/benchmarks/video/capture_camera_feed.py
 M so100_evo1/lerobot-main/docs/source/contributing.md
 M so100_evo1/lerobot-main/src/lerobot/policies/act/README.md
 M so100_evo1/lerobot-main/src/lerobot/policies/diffusion/README.md
 M so100_evo1/lerobot-main/src/lerobot/policies/evo1/dataset/config.yaml
 M so100_evo1/lerobot-main/src/lerobot/policies/evo1/ds_config.json
 M so100_evo1/lerobot-main/src/lerobot/policies/evo1/model/action_head/flow_matching.py
 M so100_evo1/lerobot-main/src/lerobot/policies/evo1/scripts/Evo1_server.py
 M so100_evo1/lerobot-main/src/lerobot/policies/smolvla/README.md
 M so100_evo1/lerobot-main/src/lerobot/polici

In [4]:
# 3. Install micromamba and create envs
MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"

if not os.path.exists(MAMBA):
    !wget -qO /tmp/micromamba.tar.bz2 https://micro.mamba.pm/api/micromamba/linux-64/latest
    !mkdir -p /content/micromamba
    !tar -xjf /tmp/micromamba.tar.bz2 -C /content/micromamba bin/micromamba

!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} create -y -n Evo1 python=3.10 pip -c conda-forge || true
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} create -y -n libero python=3.8.13 pip -c conda-forge || true
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} env list


Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                                                     ✔ Done (0.1 sec)
Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                                                     ✔ Done (0.1 sec)

Resolving Environment                                                                     ✔ Done (0.2 sec)

Transaction

  Prefix: /content/micromamba-root/envs/Evo1

  Updating specs:

   - python=3.10
   - pip


  Package               Version  Build                 Channel           Size
─────────────────────────────────────────────────────────────────

In [5]:
# 4. Install Evo-1 server deps + Blackwell-safe torch/torchao, then remove stale flash-attn
MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"
EVO = "/content/drive/MyDrive/Evo-1/Evo_1"

!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n Evo1 python -m pip install -U pip setuptools wheel
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n Evo1 python -m pip install -r "{EVO}/requirements.txt"
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n Evo1 python -m pip install --force-reinstall "huggingface-hub==0.36.2"

import subprocess, os

# Detect host GPU. Blackwell/sm_120 needs a PyTorch build that actually supports sm_120.
import torch as _host_torch
if _host_torch.cuda.is_available():
    _cc = _host_torch.cuda.get_device_capability(0)
    _gpu = _host_torch.cuda.get_device_name(0)
    print("Host GPU for Evo1 env setup:", _gpu, "capability:", _cc)
else:
    _cc = (0, 0)
    print("Host CUDA not available while setting up Evo1 env.")

if _cc[0] >= 12:
    print("Blackwell/sm_120 detected: installing PyTorch CUDA 12.8 wheels inside Evo1 env.")
    !MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n Evo1 python -m pip install --upgrade --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
else:
    print("Non-Blackwell GPU detected: keeping torch from Evo-1 requirements unless already overridden.")

# IMPORTANT: if torch changed, any already-installed flash-attn binary is now ABI-risky.
# Remove it here. The next cell rebuilds/reinstalls flash-attn against the CURRENT torch and import-tests it.
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n Evo1 python -m pip uninstall -y flash-attn flash_attn || true

# Use official PyTorch AO torchao package for W8A16 weight-only quantization, not a homemade Linear wrapper.
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n Evo1 python -m pip install --upgrade torchao --extra-index-url https://download.pytorch.org/whl/cu128

check_code = '''
import transformers, huggingface_hub, torch
import torchao
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("torch:", torch.__version__)
print("torch CUDA:", torch.version.cuda)
print("torchao:", getattr(torchao, "__version__", "unknown"))
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))
try:
    import flash_attn
    raise RuntimeError("flash_attn is still importable after uninstall. This is unsafe; rebuild cell must start from a clean state.")
except ModuleNotFoundError:
    print("flash_attn clean state: not installed yet")
'''

subprocess.run(
    [MAMBA, "run", "-n", "Evo1", "python", "-c", check_code],
    env={**os.environ, "MAMBA_ROOT_PREFIX": MAMBA_ROOT},
    check=True,
)


  Using cached transformers-4.39.0-py3-none-any.whl.metadata (134 kB)
  Using cached timm-1.0.27-py3-none-any.whl.metadata (40 kB)
  Using cached torch-2.5.1-cp310-cp310-manylinux1_x86_64.whl.metadata (28 kB)
  Using cached torchvision-0.20.1-cp310-cp310-manylinux1_x86_64.whl.metadata (6.1 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached einop-0.0.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached diffusers-0.38.0-py3-none-any.whl.metadata (20 kB)
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached matplotlib-3.10.9-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached pyarrow-24.0.0-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached accelerator-2025.11.11-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (3

CompletedProcess(args=['/content/micromamba/bin/micromamba', 'run', '-n', 'Evo1', 'python', '-c', '\nimport transformers, huggingface_hub, torch\nimport torchao\nprint("transformers:", transformers.__version__)\nprint("huggingface_hub:", huggingface_hub.__version__)\nprint("torch:", torch.__version__)\nprint("torch CUDA:", torch.version.cuda)\nprint("torchao:", getattr(torchao, "__version__", "unknown"))\nprint("CUDA:", torch.cuda.is_available())\nif torch.cuda.is_available():\n    print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))\ntry:\n    import flash_attn\n    raise RuntimeError("flash_attn is still importable after uninstall. This is unsafe; rebuild cell must start from a clean state.")\nexcept ModuleNotFoundError:\n    print("flash_attn clean state: not installed yet")\n'], returncode=0)

In [6]:
# 5. Rebuild/import-test FlashAttention against the CURRENT Evo1 torch ABI
# This fixes the undefined-symbol crash from stale flash_attn_2_cuda.so after changing torch/CUDA.
MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"
EVO = "/content/drive/MyDrive/Evo-1/Evo_1"

import os, subprocess

flash_rebuild_script = r'''
set -euxo pipefail
python - <<'PY'
import torch
print('FLASH_REBUILD_USING_TORCH:', torch.__version__)
print('FLASH_REBUILD_TORCH_CUDA:', torch.version.cuda)
print('FLASH_REBUILD_CUDA_AVAILABLE:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('FLASH_REBUILD_GPU:', torch.cuda.get_device_name(0), 'capability:', torch.cuda.get_device_capability(0))
PY

# Always remove old extension first. A stale .so causes the exact undefined-symbol import crash.
python -m pip uninstall -y flash-attn flash_attn || true
python - <<'PY'
try:
    import flash_attn
    raise SystemExit('ERROR: flash_attn still importable after uninstall')
except ModuleNotFoundError:
    print('FLASH_ATTN_UNINSTALLED_CLEANLY')
PY

# Build/install against the CURRENT torch. No cache, so pip cannot reuse a torch2.5/old-CUDA wheel.
export MAX_JOBS=4
export FLASH_ATTENTION_FORCE_BUILD=TRUE
export TORCH_CUDA_ARCH_LIST="8.0;8.6;8.9;9.0;12.0"
python -m pip install -v --no-build-isolation --no-cache-dir flash-attn

# Hard import test. The server is not allowed to start unless this passes.
python - <<'PY'
import torch
import flash_attn
import flash_attn_2_cuda
print('FLASH_ATTN_IMPORT_OK:', getattr(flash_attn, '__version__', 'unknown'))
print('FLASH_ATTN_TORCH:', torch.__version__, 'cuda', torch.version.cuda)
PY
'''

r = subprocess.run(
    [MAMBA, "run", "-n", "Evo1", "bash", "-lc", flash_rebuild_script],
    cwd=EVO,
    env={**os.environ, "MAMBA_ROOT_PREFIX": MAMBA_ROOT},
    text=True,
)
if r.returncode != 0:
    # Do not leave a half-installed broken flash-attn around. That broken package is what crashed modeling_llama import.
    cleanup = "python -m pip uninstall -y flash-attn flash_attn || true"
    subprocess.run([MAMBA, "run", "-n", "Evo1", "bash", "-lc", cleanup], env={**os.environ, "MAMBA_ROOT_PREFIX": MAMBA_ROOT}, check=False)
    raise RuntimeError(
        "FlashAttention rebuild/import failed against the current Evo1 torch. "
        "Broken flash-attn was uninstalled so the undefined-symbol crash cannot persist. "
        "Do not start the server until this cell passes."
    )
print("FLASH_ATTENTION_ABI_REBUILD_PASS")


FLASH_ATTENTION_ABI_REBUILD_PASS


In [7]:
# 6. Install LIBERO env
MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"
LIBERO_EVAL = "/content/drive/MyDrive/Evo-1/LIBERO_evaluation"

!cd "{LIBERO_EVAL}" && test -d LIBERO || git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install -U "pip<25.1" "setuptools<76" wheel
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install "numpy<1.24" "protobuf<4"
!cd "{LIBERO_EVAL}/LIBERO" && MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install -r requirements.txt
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0 --extra-index-url https://download.pytorch.org/whl/cu113
!cd "{LIBERO_EVAL}/LIBERO" && MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install -e .
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install websockets==13.1 huggingface_hub imageio imageio-ffmpeg opencv-python


  Using cached pip-25.0.1-py3-none-any.whl.metadata (3.7 kB)
  Using cached setuptools-75.3.4-py3-none-any.whl.metadata (6.9 kB)
Using cached pip-25.0.1-py3-none-any.whl (1.8 MB)
Using cached setuptools-75.3.4-py3-none-any.whl (1.3 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.3.0
    Uninstalling setuptools-75.3.0:
      Successfully uninstalled setuptools-75.3.0
  Attempting uninstall: pip
    Found existing installation: pip 24.3.1
    Uninstalling pip-24.3.1:
      Successfully uninstalled pip-24.3.1
  Using cached numpy-1.23.5-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.3 kB)
  Using cached protobuf-3.20.3-cp38-cp38-manylinux_2_5_x86_64.manylinux1_x86_64.whl.metadata (679 bytes)
Using cached numpy-1.23.5-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.1 MB)
Using cached protobuf-3.20.3-cp38-cp38-manylinux_2_5_x86_64.manylinux1_x86_64.whl (1.0 MB)
  Using cached hydra_core-1.2.0-py3-none-any.whl.metadata 

In [8]:
# 7. Download checkpoint
from huggingface_hub import snapshot_download
import os
CKPT_DIR = "/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO"
os.makedirs(CKPT_DIR, exist_ok=True)
snapshot_download(repo_id="MINT-SJTU/Evo1_LIBERO", local_dir=CKPT_DIR, local_dir_use_symlinks=False)
!find "{CKPT_DIR}" -maxdepth 2 -type f | sort | head -50


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO/checkpoint.json
/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO/config.json
/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO/mp_rank_00_model_states.pt
/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO/norm_stats.json


In [9]:
# 8. Patch server only: checkpoint path, port 9010, websocket no-timeout
from pathlib import Path
import re

REPO = "/content/drive/MyDrive/Evo-1"
EVO = f"{REPO}/Evo_1"
CKPT_DIR = "/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO"

server_path = Path(f"{EVO}/scripts/Evo1_server.py")
txt = server_path.read_text()

txt = re.sub(
    r"ckpt_dir\s*=\s*['\"].*?['\"]",
    f'ckpt_dir = "{CKPT_DIR}"',
    txt,
    count=1,
)

txt = txt.replace("9000", "9010")

if "ping_interval=None" not in txt:
    txt = txt.replace(
        'websockets.serve(handler, "0.0.0.0", PORT)',
        'websockets.serve(handler, "0.0.0.0", PORT, ping_interval=None, ping_timeout=None, close_timeout=30)',
    )
    txt = txt.replace(
        "websockets.serve(handler, '0.0.0.0', PORT)",
        "websockets.serve(handler, '0.0.0.0', PORT, ping_interval=None, ping_timeout=None, close_timeout=30)",
    )

server_path.write_text(txt)

!grep -n "ckpt_dir\|9010\|9000\|websockets.serve\|ping_interval" "{server_path}" | tail -40

63:def load_model_and_normalizer(ckpt_dir):
64:    config = json.load(open(os.path.join(ckpt_dir, "config.json")))
65:    stats = json.load(open(os.path.join(ckpt_dir, "norm_stats.json")))
72:    ckpt_path = os.path.join(ckpt_dir, "mp_rank_00_model_states.pt")
149:    ckpt_dir = "/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO"
150:    #Example: ckpt_dir = "/home/dell/checkpoints/Evo1/Evo1_MetaWorld/"
152:    port = 9010
155:    model, normalizer = load_model_and_normalizer(ckpt_dir)
159:        async with websockets.serve(


In [10]:
%%bash
# 9. LIBERO config

mkdir -p ~/.libero
mkdir -p /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/datasets

cat > ~/.libero/config.yaml <<'EOF'
benchmark_root: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero
bddl_files: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/bddl_files
init_states: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/init_files
datasets: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/datasets
assets: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/assets
EOF

cat ~/.libero/config.yaml

benchmark_root: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero
bddl_files: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/bddl_files
init_states: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/init_files
datasets: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/datasets
assets: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/assets


In [11]:
# 10. Create runtime single-episode LIBERO client copy — fixed clean version
from pathlib import Path
import re

LIBERO_EVAL = "/content/drive/MyDrive/Evo-1/LIBERO_evaluation"
src = Path(f"{LIBERO_EVAL}/libero_client_4tasks.py")
dst = Path(f"{LIBERO_EVAL}/libero_client_single_episode_runtime.py")

original = src.read_text()

prefix = """
import os

# ---- Runtime single-episode controls inserted by Colab notebook ----
SINGLE_TASK_ID = int(os.environ.get("SINGLE_TASK_ID", "0"))
SINGLE_EP_INDEX = int(os.environ.get("SINGLE_EP_INDEX", "0"))
SINGLE_SUITE = os.environ.get("SINGLE_SUITE", "libero_spatial")
SINGLE_MAX_STEPS = int(os.environ.get("SINGLE_MAX_STEPS", "25"))
SINGLE_CKPT_NAME = os.environ.get(
    "SINGLE_CKPT_NAME",
    f"Evo1_FP16_{SINGLE_SUITE}_task{SINGLE_TASK_ID}_ep{SINGLE_EP_INDEX}",
)
# -------------------------------------------------------------------
"""

txt = prefix + "\n" + original

txt = re.sub(
    r"SERVER_URL\s*=\s*['\"].*?['\"]",
    'SERVER_URL = "ws://127.0.0.1:9010"',
    txt,
    count=1,
)

txt = re.sub(r"horizon\s*=\s*\d+", "horizon = 14", txt, count=1)
txt = re.sub(r"max_steps\s*=\s*\[[^\]]+\]", "max_steps = [SINGLE_MAX_STEPS]", txt, count=1)
txt = re.sub(r"task_suites\s*=\s*\[[^\]]+\]", "task_suites = [SINGLE_SUITE]", txt, count=1)
txt = re.sub(r"num_episodes\s*=\s*\d+", "num_episodes = 1", txt, count=1)
txt = re.sub(r"ckpt_name\s*=\s*f?['\"].*?['\"]", "ckpt_name = SINGLE_CKPT_NAME", txt, count=1)

txt = txt.replace("for task_id in range(num_tasks_in_suite):", "for task_id in [SINGLE_TASK_ID]:")
txt = txt.replace("for task_id in range(min(num_tasks_in_suite, 1)):", "for task_id in [SINGLE_TASK_ID]:")
txt = txt.replace("for task_id in range(min(num_tasks_in_suite, 10)):", "for task_id in [SINGLE_TASK_ID]:")

txt = txt.replace("initial_states[episode_id]", "initial_states[SINGLE_EP_INDEX]")
txt = txt.replace("init_states[episode_id]", "init_states[SINGLE_EP_INDEX]")
txt = txt.replace("for episode_id in range(num_episodes):", "for episode_id in [0]:")

txt = txt.replace(
    "async with websockets.connect(SERVER_URL) as ws:",
    "async with websockets.connect(SERVER_URL, max_size=100_000_000, ping_interval=None, ping_timeout=None, close_timeout=30) as ws:",
)

dst.write_text(txt)

print("Wrote:", dst)
print("\nTop of generated runtime client:")
print("\n".join(dst.read_text().splitlines()[:20]))

# Remove failed marker from the previous crashed run, so Cell 15 reruns it.
RESULTS = Path("/content/drive/MyDrive/Evo-1-results/fp16_resumable")
for name in [
    "fp16_resumable_debug_libero_spatial_task0_ep0.log",
    "fp16_resumable_debug_libero_spatial_task0_ep0.done.json",
]:
    p = RESULTS / name
    if p.exists():
        p.unlink()
        print("deleted failed previous file:", p)

Wrote: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/libero_client_single_episode_runtime.py

Top of generated runtime client:

import os

# ---- Runtime single-episode controls inserted by Colab notebook ----
SINGLE_TASK_ID = int(os.environ.get("SINGLE_TASK_ID", "0"))
SINGLE_EP_INDEX = int(os.environ.get("SINGLE_EP_INDEX", "0"))
SINGLE_SUITE = os.environ.get("SINGLE_SUITE", "libero_spatial")
SINGLE_MAX_STEPS = int(os.environ.get("SINGLE_MAX_STEPS", "25"))
SINGLE_CKPT_NAME = os.environ.get(
    "SINGLE_CKPT_NAME",
    f"Evo1_FP16_{SINGLE_SUITE}_task{SINGLE_TASK_ID}_ep{SINGLE_EP_INDEX}",
)
# -------------------------------------------------------------------

import asyncio
import websockets
import numpy as np
import json
import pathlib
import os


In [12]:

# QVLA0. Paths, constants, scope knob, and hard port-kill helper.
from pathlib import Path
import os, re, json, time, subprocess, signal, shutil, textwrap

REPO = Path("/content/drive/MyDrive/Evo-1")
EVO = REPO / "Evo_1"
SCRIPTS = EVO / "scripts"
LIBERO_EVAL = REPO / "LIBERO_evaluation"

QUANT_SCOPE = "action_only"  # valid: "both", "llm_only", "action_only", "none"
assert QUANT_SCOPE in {"both", "llm_only", "action_only", "none"}

RESULTS = Path(f"/content/drive/MyDrive/Evo-1-results/torchao_w8a16_{QUANT_SCOPE}_atm_ohb")
RESULTS.mkdir(parents=True, exist_ok=True)

DRIVE_CKPT = Path("/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO")
LOCAL_CKPT = Path("/content/Evo1_LIBERO")
MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"
PORT = 9010

SERVER_LOG = Path(f"/content/evo1_torchao_w8a16_{QUANT_SCOPE}_atm_ohb_server.log")
SERVER_SCRIPT = SCRIPTS / f"Evo1_server_torchao_w8a16_{QUANT_SCOPE}_atm_ohb.py"

SCALES_DIR = Path(f"/content/drive/MyDrive/Evo-1-results/quantvla_scales/evo1_torchao_w8a16_{QUANT_SCOPE}_atm_ohb_v1")
SCALES_DIR.mkdir(parents=True, exist_ok=True)
SCALES_PATH = SCALES_DIR / "scales.json"
SCALES_META_PATH = SCALES_DIR / "scales_meta.json"
DIAG_PATH = SCALES_DIR / "calib_diagnostics.json"
BUFFER_PATH = SCALES_DIR / "fixed_calib_buffer.jsonl"

TAG = f"torchao_w8a16_{QUANT_SCOPE}_atm_ohb_spatial_task0_4ep"
CALIB_TAG = f"torchao_w8a16_{QUANT_SCOPE}_calib_fixed_spatial_task0"
CALIB_REQUESTS = 32
CALIB_EPISODES = list(range(12))
TASK_SUITES = ["libero_spatial"]
TASK_IDS = [0]
EPISODES = list(range(4))
MAX_STEPS_BY_SUITE = {"libero_spatial": 25, "libero_object": 25, "libero_goal": 25, "libero_10": 25}

# QuantVLA-style mechanisms are ON by default. For ablation only, set APPLY_ATM="0" manually.
# Note: q/k ATM gain changes attention-logit scale, so keep the printed diagnostics and compare carefully.
APPLY_CONTEXT_GAIN = "1"
APPLY_ATM = "1"
APPLY_OHB_ATTN = "1"
APPLY_OHB_FF = "1"

FORCE_RECALIB = False

print("REPO:", REPO)
print("SERVER_SCRIPT:", SERVER_SCRIPT)
print("SERVER_LOG:", SERVER_LOG)
print("RESULTS:", RESULTS)
print("SCALES_PATH:", SCALES_PATH)
print("QUANT_BACKEND: torchao.Int8WeightOnlyConfig")
print("QUANT_SCOPE:", QUANT_SCOPE)
print("APPLY_ATM:", APPLY_ATM, "(ON by default; set 0 only for ablation)")

def run(cmd, **kwargs):
    print("+", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    return subprocess.run(cmd, shell=isinstance(cmd, str), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kwargs)

def kill_port_9010():
    r = run("ss -ltnp | grep ':9010' || true")
    txt = r.stdout.strip()
    if not txt:
        print("no process currently listening on 9010")
        return
    print(txt)
    pids = sorted(set(re.findall(r"pid=(\d+)", txt)))
    for pid in pids:
        print("killing pid", pid)
        subprocess.run(["kill", "-9", pid], check=False)
    time.sleep(2)
    r2 = run("ss -ltnp | grep ':9010' || true")
    if r2.stdout.strip():
        raise RuntimeError("port 9010 still busy after kill:\n" + r2.stdout)
    print("port 9010 is free")


REPO: /content/drive/MyDrive/Evo-1
SERVER_SCRIPT: /content/drive/MyDrive/Evo-1/Evo_1/scripts/Evo1_server_torchao_w8a16_action_only_atm_ohb.py
SERVER_LOG: /content/evo1_torchao_w8a16_action_only_atm_ohb_server.log
RESULTS: /content/drive/MyDrive/Evo-1-results/torchao_w8a16_action_only_atm_ohb
SCALES_PATH: /content/drive/MyDrive/Evo-1-results/quantvla_scales/evo1_torchao_w8a16_action_only_atm_ohb_v1/scales.json
QUANT_BACKEND: torchao.Int8WeightOnlyConfig
QUANT_SCOPE: action_only
APPLY_ATM: 1 (ON by default; set 0 only for ablation)


In [13]:

# QVLA1. Copy checkpoint locally and verify torchao backend imports in Evo1 env.
import subprocess, shutil, os, re
from pathlib import Path

assert REPO.exists(), f"Missing Evo-1 repo: {REPO}"
assert EVO.exists(), f"Missing Evo_1 folder: {EVO}"
assert DRIVE_CKPT.exists(), f"Missing checkpoint folder: {DRIVE_CKPT}"

LOCAL_CKPT.mkdir(parents=True, exist_ok=True)
subprocess.run(["rsync", "-ah", "--info=progress2", str(DRIVE_CKPT) + "/", str(LOCAL_CKPT) + "/"], check=True)
print("Using LOCAL_CKPT:", LOCAL_CKPT)

check_code = r"""
import torch
import torchao
from torchao.quantization import quantize_, Int8WeightOnlyConfig
try:
    from torchao.quantization import AffineQuantizedTensor
except Exception:
    try:
        from torchao.dtypes import AffineQuantizedTensor
    except Exception:
        from torchao.dtypes.affine_quantized_tensor import AffineQuantizedTensor
print("torch:", torch.__version__)
print("torchao:", getattr(torchao, "__version__", "unknown"))
print("Int8WeightOnlyConfig:", Int8WeightOnlyConfig)
print("AffineQuantizedTensor:", AffineQuantizedTensor)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))
"""
subprocess.run([MAMBA, "run", "-n", "Evo1", "python", "-c", check_code],
               env={**os.environ, "MAMBA_ROOT_PREFIX": MAMBA_ROOT}, check=True)
print("TORCHAO_BACKEND_IMPORT_PASS")


Using LOCAL_CKPT: /content/Evo1_LIBERO
TORCHAO_BACKEND_IMPORT_PASS


In [14]:
# QVLA2. Create runtime single-episode LIBERO client copy — preserve baseline client structure
from pathlib import Path
import re

LIBERO_EVAL = "/content/drive/MyDrive/Evo-1/LIBERO_evaluation"
src = Path(f"{LIBERO_EVAL}/libero_client_4tasks.py")
dst = Path(f"{LIBERO_EVAL}/libero_client_single_episode_runtime.py")

original = src.read_text()

prefix = """
import os

# ---- Runtime single-episode controls inserted by Colab notebook ----
SINGLE_TASK_ID = int(os.environ.get("SINGLE_TASK_ID", "0"))
SINGLE_EP_INDEX = int(os.environ.get("SINGLE_EP_INDEX", "0"))
SINGLE_SUITE = os.environ.get("SINGLE_SUITE", "libero_spatial")
SINGLE_MAX_STEPS = int(os.environ.get("SINGLE_MAX_STEPS", "25"))
SINGLE_CKPT_NAME = os.environ.get(
    "SINGLE_CKPT_NAME",
    f"Evo1_FP16_{SINGLE_SUITE}_task{SINGLE_TASK_ID}_ep{SINGLE_EP_INDEX}",
)
# -------------------------------------------------------------------
"""

txt = prefix + "\n" + original

txt = re.sub(
    r"SERVER_URL\s*=\s*['\"].*?['\"]",
    'SERVER_URL = "ws://127.0.0.1:9010"',
    txt,
    count=1,
)

txt = re.sub(r"horizon\s*=\s*\d+", "horizon = 14", txt, count=1)
txt = re.sub(r"max_steps\s*=\s*\[[^\]]+\]", "max_steps = [SINGLE_MAX_STEPS]", txt, count=1)
txt = re.sub(r"task_suites\s*=\s*\[[^\]]+\]", "task_suites = [SINGLE_SUITE]", txt, count=1)
txt = re.sub(r"num_episodes\s*=\s*\d+", "num_episodes = 1", txt, count=1)
txt = re.sub(r"ckpt_name\s*=\s*f?['\"].*?['\"]", "ckpt_name = SINGLE_CKPT_NAME", txt, count=1)

txt = txt.replace("for task_id in range(num_tasks_in_suite):", "for task_id in [SINGLE_TASK_ID]:")
txt = txt.replace("for task_id in range(min(num_tasks_in_suite, 1)):", "for task_id in [SINGLE_TASK_ID]:")
txt = txt.replace("for task_id in range(min(num_tasks_in_suite, 10)):", "for task_id in [SINGLE_TASK_ID]:")

txt = txt.replace("initial_states[episode_id]", "initial_states[SINGLE_EP_INDEX]")
txt = txt.replace("init_states[episode_id]", "init_states[SINGLE_EP_INDEX]")
txt = txt.replace("for episode_id in range(num_episodes):", "for episode_id in [0]:")

txt = txt.replace(
    "async with websockets.connect(SERVER_URL) as ws:",
    "async with websockets.connect(SERVER_URL, max_size=100_000_000, ping_interval=None, ping_timeout=None, close_timeout=30) as ws:",
)

dst.write_text(txt)

print("Wrote:", dst)
print("\nTop of generated runtime client:")
print("\n".join(dst.read_text().splitlines()[:20]))

# Remove failed marker from the previous crashed run, so Cell 15 reruns it.
print("Runtime client created; no old result deletion in QVLA notebook.")


Wrote: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/libero_client_single_episode_runtime.py

Top of generated runtime client:

import os

# ---- Runtime single-episode controls inserted by Colab notebook ----
SINGLE_TASK_ID = int(os.environ.get("SINGLE_TASK_ID", "0"))
SINGLE_EP_INDEX = int(os.environ.get("SINGLE_EP_INDEX", "0"))
SINGLE_SUITE = os.environ.get("SINGLE_SUITE", "libero_spatial")
SINGLE_MAX_STEPS = int(os.environ.get("SINGLE_MAX_STEPS", "25"))
SINGLE_CKPT_NAME = os.environ.get(
    "SINGLE_CKPT_NAME",
    f"Evo1_FP16_{SINGLE_SUITE}_task{SINGLE_TASK_ID}_ep{SINGLE_EP_INDEX}",
)
# -------------------------------------------------------------------

import asyncio
import websockets
import numpy as np
import json
import pathlib
import os
Runtime client created; no old result deletion in QVLA notebook.


### PATCH NOTE — torchao W8A16 critical fixes

This notebook version fixes three concrete runtime bugs:

1. Server now prints `[EVO1-W8A16] W8A16-CALIB-DONE`, matching the calibration wait loop. The old `[EVO1-W8A16] CALIB-DONE` marker is also printed for compatibility.
2. `load_model_and_normalizer()` now moves Evo-1 to CUDA **and BF16** before `torchao.quantize_()`: `model.to("cuda").to(torch.bfloat16)`.
3. torchao replacement verification now uses `isinstance(module.weight, AffineQuantizedTensor)`, not a fragile string match on type names.


### PATCH NOTE — torchao Int8Tensor verification

Verification now accepts the actual `torchao` `Int8Tensor` produced by `Int8WeightOnlyConfig`, with `AffineQuantizedTensor` compatibility fallback. This avoids both the old string-name hack and the wrong single-class check.

In [15]:
# QVLA3. Write Evo-1 torchao W8A16 all-linear diagnostic server script.
SERVER_SCRIPT.write_text('# Evo-1 server: FP16 teacher calibration + torchao W8A16 static weight-only student + QuantVLA-style diagnostics.\n# Quant backend is PyTorch AO / torchao Int8WeightOnlyConfig. No handwritten W8A16 Linear implementation.\nimport sys, os, asyncio, websockets, numpy as np, cv2, json, torch, math, time, gc, re\nfrom pathlib import Path\nfrom PIL import Image\nfrom torchvision import transforms\nimport torch.nn.functional as F\n\nsys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))\nfrom scripts.Evo1 import EVO1\n\ntry:\n    import importlib\n    import torchao\n    from torchao.quantization import quantize_, Int8WeightOnlyConfig\n    _TORCHAO_INT8_WEIGHT_TYPES = []\n    # Int8WeightOnlyConfig currently stores Linear.weight as a torchao Int8Tensor.\n    # Some older/alternate torchao paths expose AffineQuantizedTensor. Accept actual\n    # tensor classes and never verify by fragile string-name matching.\n    for _mod_name, _cls_name in [\n        ("torchao.quantization.quantize_.workflows", "Int8Tensor"),\n        ("torchao.dtypes", "Int8Tensor"),\n        ("torchao.quantization", "AffineQuantizedTensor"),\n        ("torchao.dtypes", "AffineQuantizedTensor"),\n        ("torchao.dtypes.affine_quantized_tensor", "AffineQuantizedTensor"),\n    ]:\n        try:\n            _cls = getattr(importlib.import_module(_mod_name), _cls_name)\n            if _cls not in _TORCHAO_INT8_WEIGHT_TYPES:\n                _TORCHAO_INT8_WEIGHT_TYPES.append(_cls)\n                print(f"[EVO1-W8A16] TORCHAO-QTENSOR-TYPE: {_mod_name}.{_cls_name}", flush=True)\n        except Exception:\n            pass\n    if not _TORCHAO_INT8_WEIGHT_TYPES:\n        raise RuntimeError("Could not resolve torchao Int8Tensor/AffineQuantizedTensor class for verification")\n    _TORCHAO_INT8_WEIGHT_TYPES = tuple(_TORCHAO_INT8_WEIGHT_TYPES)\nexcept Exception as e:\n    print("[EVO1-W8A16][FATAL] Could not import torchao Int8WeightOnlyConfig/quantized tensor types:", repr(e), flush=True)\n    raise\n\n\nclass Normalizer:\n    def __init__(self, stats_or_path):\n        if isinstance(stats_or_path, str):\n            with open(stats_or_path, "r") as f:\n                stats = json.load(f)\n        else:\n            stats = stats_or_path\n        def pad_to_24(x):\n            x = torch.tensor(x, dtype=torch.float32)\n            if x.shape[0] < 24:\n                x = torch.cat([x, torch.zeros(24 - x.shape[0], dtype=torch.float32)], dim=0)\n            elif x.shape[0] > 24:\n                raise ValueError(f"Input length {x.shape[0]} exceeds expected 24")\n            return x\n        if len(stats) != 1:\n            raise ValueError(f"norm_stats.json should contain one robot key, got {list(stats.keys())}")\n        robot_stats = stats[list(stats.keys())[0]]\n        self.state_min = pad_to_24(robot_stats["observation.state"]["min"])\n        self.state_max = pad_to_24(robot_stats["observation.state"]["max"])\n        self.action_min = pad_to_24(robot_stats["action"]["min"])\n        self.action_max = pad_to_24(robot_stats["action"]["max"])\n    def normalize_state(self, state):\n        state_min = self.state_min.to(state.device, dtype=state.dtype)\n        state_max = self.state_max.to(state.device, dtype=state.dtype)\n        return torch.clamp(2 * (state - state_min) / (state_max - state_min + 1e-8) - 1, -1.0, 1.0)\n    def denormalize_action(self, action):\n        action_min = self.action_min.to(action.device, dtype=action.dtype)\n        action_max = self.action_max.to(action.device, dtype=action.dtype)\n        if action.ndim == 1:\n            action = action.view(1, -1)\n        return (action + 1.0) / 2.0 * (action_max - action_min + 1e-8) + action_min\n\ndef load_model_and_normalizer(ckpt_dir):\n    config = json.load(open(os.path.join(ckpt_dir, "config.json")))\n    stats = json.load(open(os.path.join(ckpt_dir, "norm_stats.json")))\n    config["finetune_vlm"] = False\n    config["finetune_action_head"] = False\n    config["num_inference_timesteps"] = int(os.environ.get("EVO1_NUM_INFERENCE_TIMESTEPS", "32"))\n    model = EVO1(config).eval()\n    ckpt_path = os.path.join(ckpt_dir, "mp_rank_00_model_states.pt")\n    checkpoint = torch.load(ckpt_path, map_location="cpu")\n    model.load_state_dict(checkpoint["module"], strict=True)\n    # Keep Evo-1 model dtype aligned with the established W8A8/QuantVLA-style flow.\n    # torchao then quantizes selected Linear weights from this BF16 model; activations stay BF16/FP16 under autocast.\n    model = model.to("cuda").to(torch.bfloat16)\n    print("[EVO1-W8A16] MODEL_DTYPE_AFTER_LOAD: bfloat16", flush=True)\n    return model, Normalizer(stats)\n\ndef decode_image_from_list(img_list):\n    img_array = np.array(img_list, dtype=np.uint8)\n    img = cv2.resize(img_array, (448, 448))\n    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)\n    pil = Image.fromarray(img)\n    return transforms.ToTensor()(pil).to("cuda")\n\ndef prepare_inputs(data, normalizer):\n    device = "cuda"\n    images = [decode_image_from_list(img) for img in data["image"]]\n    assert len(images) == 3, "Must provide exactly 3 images."\n    state = torch.tensor(data["state"], dtype=torch.float32, device=device)\n    if state.ndim == 1:\n        state = state.unsqueeze(0)\n    if state.shape[1] < 24:\n        state = torch.cat([state, torch.zeros((1, 24 - state.shape[1]), device=device)], dim=1)\n    norm_state = normalizer.normalize_state(state).to(dtype=torch.float32)\n    prompt = data["prompt"]\n    image_mask = torch.tensor(data["image_mask"], dtype=torch.int32, device=device)\n    action_mask = torch.tensor([data["action_mask"]], dtype=torch.int32, device=device)\n    return images, norm_state, prompt, image_mask, action_mask\n\nclass QVLAStats:\n    def __init__(self, num_layers=8, num_heads=8):\n        self.num_layers = num_layers\n        self.num_heads = num_heads\n        self.reset()\n    def reset(self):\n        self.n_context = 0\n        self.ctx_fp_sq = None\n        self.ctx_q_sq = None\n        self.ctx_dot = 0.0\n        self.ctx_fp_norm = 0.0\n        self.ctx_q_norm = 0.0\n        self.layer = {\n            role: {\n                "logit_std_sum": torch.zeros(self.num_layers, self.num_heads, dtype=torch.float64),\n                "attn_rms_sum": torch.zeros(self.num_layers, dtype=torch.float64),\n                "ff_rms_sum": torch.zeros(self.num_layers, dtype=torch.float64),\n                "count": torch.zeros(self.num_layers, dtype=torch.float64),\n            } for role in ["teacher", "student"]\n        }\n    def record_context(self, fp, q):\n        with torch.no_grad():\n            fp2 = fp.detach().float().reshape(-1, fp.shape[-1]).cpu()\n            q2 = q.detach().float().reshape(-1, q.shape[-1]).cpu()\n            if self.ctx_fp_sq is None:\n                self.ctx_fp_sq = torch.zeros(fp2.shape[-1], dtype=torch.float64)\n                self.ctx_q_sq = torch.zeros(q2.shape[-1], dtype=torch.float64)\n            self.ctx_fp_sq += (fp2.double() ** 2).sum(dim=0)\n            self.ctx_q_sq += (q2.double() ** 2).sum(dim=0)\n            self.ctx_dot += float((fp2.double() * q2.double()).sum())\n            self.ctx_fp_norm += float((fp2.double() ** 2).sum())\n            self.ctx_q_norm += float((q2.double() ** 2).sum())\n            self.n_context += fp2.shape[0]\n    def record_block(self, role, idx, logit_std, attn_out, ff_out):\n        with torch.no_grad():\n            d = self.layer[role]\n            ls = logit_std.detach().float().mean(dim=0).cpu().double()  # [heads]\n            d["logit_std_sum"][idx] += ls\n            d["attn_rms_sum"][idx] += float(torch.sqrt(torch.mean(attn_out.detach().float() ** 2) + 1e-12).cpu())\n            d["ff_rms_sum"][idx] += float(torch.sqrt(torch.mean(ff_out.detach().float() ** 2) + 1e-12).cpu())\n            d["count"][idx] += 1\n    def finalize(self):\n        eps = 1e-8\n        fp_ctx_rms = torch.sqrt(self.ctx_fp_sq / max(1, self.n_context) + eps)\n        q_ctx_rms = torch.sqrt(self.ctx_q_sq / max(1, self.n_context) + eps)\n        context_gain_vec = torch.clamp(fp_ctx_rms / torch.clamp(q_ctx_rms, min=eps), 0.25, 4.0)\n        context_gain_scalar = float(torch.clamp(torch.sqrt(torch.tensor(self.ctx_fp_norm / max(self.ctx_q_norm, eps))), 0.25, 4.0))\n        context_cos = float(self.ctx_dot / math.sqrt(max(self.ctx_fp_norm, eps) * max(self.ctx_q_norm, eps)))\n        teacher = self.layer["teacher"]\n        student = self.layer["student"]\n        count = torch.clamp(student["count"], min=1.0)\n        t_count = torch.clamp(teacher["count"], min=1.0)\n        t_logit = teacher["logit_std_sum"] / t_count[:, None]\n        q_logit = student["logit_std_sum"] / count[:, None]\n        alpha = torch.clamp(t_logit / torch.clamp(q_logit, min=eps), 0.25, 4.0)\n        t_attn = teacher["attn_rms_sum"] / t_count\n        q_attn = student["attn_rms_sum"] / count\n        t_ff = teacher["ff_rms_sum"] / t_count\n        q_ff = student["ff_rms_sum"] / count\n        beta_attn = torch.clamp(t_attn / torch.clamp(q_attn, min=eps), 0.25, 4.0)\n        beta_ff = torch.clamp(t_ff / torch.clamp(q_ff, min=eps), 0.25, 4.0)\n        # Before/after ratios are the main sanity check.  Before = quant / fp.\n        # After = corrected_quant / fp using the scale that will be applied during eval.\n        context_rms_fp_mean = float(fp_ctx_rms.mean().item())\n        context_rms_q_mean = float(q_ctx_rms.mean().item())\n        context_rms_ratio_before = float(context_rms_q_mean / max(context_rms_fp_mean, eps))\n        context_rms_ratio_after_scalar = float((context_rms_q_mean * context_gain_scalar) / max(context_rms_fp_mean, eps))\n        context_rms_ratio_after_vec_mean = float(((q_ctx_rms * context_gain_vec) / torch.clamp(fp_ctx_rms, min=eps)).mean().item())\n        logit_ratio_before = q_logit / torch.clamp(t_logit, min=eps)\n        logit_ratio_after = (q_logit * alpha) / torch.clamp(t_logit, min=eps)\n        attn_ratio_before = q_attn / torch.clamp(t_attn, min=eps)\n        attn_ratio_after = (q_attn * beta_attn) / torch.clamp(t_attn, min=eps)\n        ff_ratio_before = q_ff / torch.clamp(t_ff, min=eps)\n        ff_ratio_after = (q_ff * beta_ff) / torch.clamp(t_ff, min=eps)\n        scales = {\n            "method": "Evo1 QuantVLA-style diagnostic calibration: torchao Int8WeightOnlyConfig W8A16 static weight-only + fixed unlabeled calibration buffer + context/ATM/OHB drift statistics",\n            "formula_version": "evo1_torchao_w8a16_diag_v1",\n            "formulas": {\n                "context_rms": "sqrt(mean(context^2))",\n                "context_gain_scalar": "sqrt(sum(context_fp^2) / sum(context_q^2))",\n                "context_gain_vec": "sqrt(mean(context_fp^2, dim=samples) / mean(context_q^2, dim=samples))",\n                "context_cosine": "dot(context_fp, context_q) / (||context_fp|| ||context_q||)",\n                "atm_alpha": "std(attention_logits_fp) / std(attention_logits_q)",\n                "ohb_beta_attn": "rms(attn_out_fp) / rms(attn_out_q)",\n                "ohb_beta_ff": "rms(ff_out_fp) / rms(ff_out_q)",\n                "ratio_before": "quant_stat / fp_stat",\n                "ratio_after": "corrected_quant_stat / fp_stat"\n            },\n            "context_gain_scalar": context_gain_scalar,\n            "context_gain_vec": [float(x) for x in context_gain_vec.tolist()],\n            "context_cosine_mean": context_cos,\n            "context_rms_fp_mean": context_rms_fp_mean,\n            "context_rms_q_mean": context_rms_q_mean,\n            "context_rms_ratio_before": context_rms_ratio_before,\n            "context_rms_ratio_after_scalar": context_rms_ratio_after_scalar,\n            "context_rms_ratio_after_vec_mean": context_rms_ratio_after_vec_mean,\n            "atm_alpha_per_layer_head": [[float(x) for x in row] for row in alpha.tolist()],\n            "atm_logit_ratio_before_per_layer_head": [[float(x) for x in row] for row in logit_ratio_before.tolist()],\n            "atm_logit_ratio_after_per_layer_head": [[float(x) for x in row] for row in logit_ratio_after.tolist()],\n            "atm_logit_ratio_before_mean": float(logit_ratio_before.mean().item()),\n            "atm_logit_ratio_after_mean": float(logit_ratio_after.mean().item()),\n            "ohb_beta_attn_per_layer": [float(x) for x in beta_attn.tolist()],\n            "ohb_attn_ratio_before_per_layer": [float(x) for x in attn_ratio_before.tolist()],\n            "ohb_attn_ratio_after_per_layer": [float(x) for x in attn_ratio_after.tolist()],\n            "ohb_attn_ratio_before_mean": float(attn_ratio_before.mean().item()),\n            "ohb_attn_ratio_after_mean": float(attn_ratio_after.mean().item()),\n            "ohb_beta_ff_per_layer": [float(x) for x in beta_ff.tolist()],\n            "ohb_ff_ratio_before_per_layer": [float(x) for x in ff_ratio_before.tolist()],\n            "ohb_ff_ratio_after_per_layer": [float(x) for x in ff_ratio_after.tolist()],\n            "ohb_ff_ratio_before_mean": float(ff_ratio_before.mean().item()),\n            "ohb_ff_ratio_after_mean": float(ff_ratio_after.mean().item()),\n            "teacher_logit_std": [[float(x) for x in row] for row in t_logit.tolist()],\n            "student_logit_std": [[float(x) for x in row] for row in q_logit.tolist()],\n            "teacher_attn_rms": [float(x) for x in t_attn.tolist()],\n            "student_attn_rms": [float(x) for x in q_attn.tolist()],\n            "teacher_ff_rms": [float(x) for x in t_ff.tolist()],\n            "student_ff_rms": [float(x) for x in q_ff.tolist()],\n            "context_tokens_count": int(self.n_context),\n            "action_block_forward_count_max": int(student["count"].max().item()),\n            "scale_clamp_min": 0.25,\n            "scale_clamp_max": 4.0,\n        }\n        # Basic validity flags; notebook/server abort if these are bad.\n        bad_values = []\n        for k, v in scales.items():\n            if isinstance(v, float) and (math.isnan(v) or math.isinf(v)):\n                bad_values.append(k)\n        if bad_values:\n            raise RuntimeError("Invalid calibration scale values: " + repr(bad_values))\n        return scales\n\nclass QVLAApply:\n    def __init__(self, scales=None, role="student", collect=False, stats=None):\n        self.scales = scales or {}\n        self.role = role\n        self.collect = collect\n        self.stats = stats\n        self.context_gain = None\n        self.alpha = None\n        self.beta_attn = None\n        self.beta_ff = None\n        self.apply_context_gain = os.environ.get("EVO1_QVLA_APPLY_CONTEXT_GAIN", "1") not in ("0", "false", "False")\n        self.apply_atm = os.environ.get("EVO1_QVLA_APPLY_ATM", "1") not in ("0", "false", "False")\n        self.apply_ohb_attn = os.environ.get("EVO1_QVLA_APPLY_OHB_ATTN", "1") not in ("0", "false", "False")\n        self.apply_ohb_ff = os.environ.get("EVO1_QVLA_APPLY_OHB_FF", "1") not in ("0", "false", "False")\n        if scales:\n            self.context_gain = float(scales.get("context_gain_scalar", 1.0))\n            self.alpha = torch.tensor(scales.get("atm_alpha_per_layer_head", []), dtype=torch.float32, device="cuda")\n            self.beta_attn = torch.tensor(scales.get("ohb_beta_attn_per_layer", []), dtype=torch.float32, device="cuda")\n            self.beta_ff = torch.tensor(scales.get("ohb_beta_ff_per_layer", []), dtype=torch.float32, device="cuda")\n    def correct_context(self, fused):\n        if self.scales and self.apply_context_gain:\n            return fused * self.context_gain\n        return fused\n\ndef install_qvla_action_patch(model, qvla_apply: QVLAApply):\n    # Replace each BasicTransformerBlock.forward with an equivalent explicit MHA path so ATM/OHB can be measured/applied.\n    blocks = model.action_head.transformer_blocks\n    for idx, block in enumerate(blocks):\n        attn = block.attn\n        embed_dim = attn.embed_dim\n        num_heads = attn.num_heads\n        head_dim = embed_dim // num_heads\n        def make_forward(block, idx, embed_dim, num_heads, head_dim):\n            def forward(action_tokens, context_tokens, time_emb):\n                x_norm = block.norm1(action_tokens)\n                W = block.attn.in_proj_weight\n                b = block.attn.in_proj_bias\n                if W is None:\n                    raise RuntimeError("Expected nn.MultiheadAttention with fused in_proj_weight")\n                q = F.linear(x_norm, W[:embed_dim], b[:embed_dim] if b is not None else None)\n                k = F.linear(context_tokens, W[embed_dim:2*embed_dim], b[embed_dim:2*embed_dim] if b is not None else None)\n                v = F.linear(context_tokens, W[2*embed_dim:], b[2*embed_dim:] if b is not None else None)\n                B, Lq, _ = q.shape\n                Lk = k.shape[1]\n                q = q.view(B, Lq, num_heads, head_dim).transpose(1, 2)\n                k = k.view(B, Lk, num_heads, head_dim).transpose(1, 2)\n                v = v.view(B, Lk, num_heads, head_dim).transpose(1, 2)\n                logits = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(head_dim)\n                logit_std = logits.detach().float().std(dim=(-2, -1), unbiased=False)  # [B,H]\n                if qvla_apply.scales and qvla_apply.apply_atm and qvla_apply.alpha is not None and qvla_apply.alpha.numel() > 0:\n                    logits = logits * qvla_apply.alpha[idx].view(1, num_heads, 1, 1).to(dtype=logits.dtype)\n                probs = torch.softmax(logits, dim=-1)\n                out = torch.matmul(probs, v).transpose(1, 2).contiguous().view(B, Lq, embed_dim)\n                attn_out = block.attn.out_proj(out)\n                raw_attn_out = attn_out\n                if qvla_apply.scales and qvla_apply.apply_ohb_attn and qvla_apply.beta_attn is not None and qvla_apply.beta_attn.numel() > 0:\n                    attn_out = attn_out * qvla_apply.beta_attn[idx].to(dtype=attn_out.dtype)\n                x = action_tokens + attn_out\n                x2 = block.norm2(x)\n                if time_emb is not None:\n                    x2 = x2 + time_emb.unsqueeze(1)\n                ff_out = block.ff(x2)\n                raw_ff_out = ff_out\n                if qvla_apply.scales and qvla_apply.apply_ohb_ff and qvla_apply.beta_ff is not None and qvla_apply.beta_ff.numel() > 0:\n                    ff_out = ff_out * qvla_apply.beta_ff[idx].to(dtype=ff_out.dtype)\n                if qvla_apply.collect and qvla_apply.stats is not None:\n                    qvla_apply.stats.record_block(qvla_apply.role, idx, logit_std, raw_attn_out, raw_ff_out)\n                return x + ff_out\n            return forward\n        block.forward = make_forward(block, idx, embed_dim, num_heads, head_dim)\n\n\nLLM_ALL_LINEAR_RE = re.compile(\n    r"^embedder\\.model\\.language_model\\.(?:model\\.layers|layers)\\.\\d+\\."\n    r"(?:self_attn\\.(?:q_proj|k_proj|v_proj|o_proj)|mlp\\.(?:gate_proj|up_proj|down_proj))$"\n)\nACTION_BLOCK_LINEAR_RE = re.compile(r"^action_head\\.transformer_blocks\\.\\d+\\..*$")\n\ndef _scope_parts():\n    scope = os.environ.get("EVO1_QVLA_QUANT_SCOPE", "both").strip().lower()\n    valid = {"both", "llm_only", "action_only", "none"}\n    if scope not in valid:\n        raise ValueError(f"Bad EVO1_QVLA_QUANT_SCOPE={scope!r}; valid={sorted(valid)}")\n    return scope\n\ndef expected_targets(model, quant_scope=None):\n    quant_scope = quant_scope or _scope_parts()\n    llm, action, other_linear = [], [], []\n    for n, m in model.named_modules():\n        if isinstance(m, torch.nn.Linear):\n            if LLM_ALL_LINEAR_RE.match(n):\n                llm.append(n)\n            elif ACTION_BLOCK_LINEAR_RE.match(n):\n                action.append(n)\n            elif len(other_linear) < 40:\n                other_linear.append(n)\n    target = []\n    if quant_scope in ("both", "llm_only"):\n        target.extend(llm)\n    if quant_scope in ("both", "action_only"):\n        target.extend(action)\n    return sorted(llm), sorted(action), sorted(target), other_linear\n\ndef _is_torchao_int8_weight(module):\n    # torchao Int8WeightOnlyConfig replaces Linear.weight with a torchao tensor subclass\n    # such as Int8Tensor. Some versions expose AffineQuantizedTensor wrappers.\n    # Use actual torchao classes, never a fragile string-name check.\n    w = getattr(module, "weight", None)\n    return isinstance(w, _TORCHAO_INT8_WEIGHT_TYPES)\n\ndef apply_torchao_w8a16_to_student(student):\n    quant_scope = _scope_parts()\n    llm, action, target, other = expected_targets(student, quant_scope)\n    target_set = set(target)\n    print(f"[EVO1-W8A16] QUANT_BACKEND: torchao.Int8WeightOnlyConfig", flush=True)\n    print(f"[EVO1-W8A16] QUANT_SCOPE: {quant_scope}", flush=True)\n    print(f"[EVO1-W8A16] preflight MATCH-LLM-ALL-LINEAR: {len(llm)}", flush=True)\n    print(f"[EVO1-W8A16] preflight MATCH-ACTION-BLOCK-LINEAR: {len(action)}", flush=True)\n    print(f"[EVO1-W8A16] SELECTED-TARGETS: {len(target)}", flush=True)\n    print(f"[EVO1-W8A16] EXPECTED-LLM-ALL-LINEAR: 98", flush=True)\n    print(f"[EVO1-W8A16] ACTUAL-LLM-ALL-LINEAR: {len(llm)}", flush=True)\n    if quant_scope in ("both", "llm_only") and len(llm) != 98:\n        raise RuntimeError(f"LLM all-linear target count wrong: {len(llm)} != 98")\n    if quant_scope in ("both", "action_only") and len(action) < 16:\n        raise RuntimeError(f"Action-head transformer block Linear count too small: {len(action)} < 16")\n    if quant_scope != "none" and not target:\n        raise RuntimeError("No torchao W8A16 target modules selected")\n    print(f"[EVO1-W8A16] TARGET-LAYER-LIST-BEGIN total={len(target)}", flush=True)\n    for name in llm:\n        if name in target_set:\n            print("[EVO1-W8A16][TARGET-LLM-LINEAR] " + name, flush=True)\n    for name in action:\n        if name in target_set:\n            print("[EVO1-W8A16][TARGET-ACTION-LINEAR] " + name, flush=True)\n    print(f"[EVO1-W8A16] TARGET-LAYER-LIST-END total={len(target)}", flush=True)\n\n    if quant_scope != "none":\n        def filt(mod, fqn):\n            return isinstance(mod, torch.nn.Linear) and fqn in target_set\n        quantize_(student, Int8WeightOnlyConfig(), filter_fn=filt, device="cuda")\n\n    replaced = []\n    for n, m in student.named_modules():\n        if n in target_set and _is_torchao_int8_weight(m):\n            replaced.append(n)\n    replaced = sorted(replaced)\n    print(f"[EVO1-W8A16] TORCHAO-W8A16-REPLACED: {len(replaced)}", flush=True)\n    for name in replaced:\n        print("[EVO1-W8A16][TORCHAO-INT8-WEIGHT] " + name, flush=True)\n    if quant_scope != "none" and set(replaced) != target_set:\n        missing = sorted(target_set - set(replaced))\n        extra = sorted(set(replaced) - target_set)\n        debug = []\n        modules = dict(student.named_modules())\n        for name in missing[:10]:\n            m = modules.get(name)\n            w = getattr(m, "weight", None) if m is not None else None\n            debug.append((name, type(w).__module__ + "." + type(w).__name__ if w is not None else None))\n        print(f"[EVO1-W8A16][TORCHAO-MISMATCH-DEBUG] missing_weight_types={debug}", flush=True)\n        raise RuntimeError(f"torchao target replacement mismatch; missing={missing[:20]}, extra={extra[:20]}")\n    print("[EVO1-W8A16] TORCHAO-INT8-WEIGHTONLY-PROOF: True", flush=True)\n    return {\n        "quant_scope": quant_scope,\n        "language_model_linear": len(llm) if quant_scope in ("both", "llm_only") else 0,\n        "action_head_linear": len(action) if quant_scope in ("both", "action_only") else 0,\n        "target_total": len(target),\n        "target_names": target,\n        "replaced_names": replaced,\n    }\n\n\ndef action_to_json(action, normalizer):\n    action = action.reshape(1, -1, 24)\n    action = normalizer.denormalize_action(action[0])\n    return action.cpu().numpy().tolist()\n\n\nclass QVLAServerState:\n    def __init__(self):\n        self.mode = os.environ.get("EVO1_QVLA_MODE", "calib_then_eval")\n        self.calib_requests_target = int(os.environ.get("EVO1_QVLA_CALIB_REQUESTS", "32"))\n        self.request_i = 0\n        self.calib_done = False\n        self.stats = None\n        self.scales = None\n        self.teacher = None\n        self.student = None\n        self.normalizer = None\n        self.scales_path = Path(os.environ.get("EVO1_QVLA_SCALES_PATH", "/content/drive/MyDrive/Evo-1-results/evo1_torchao_w8a16_scales.json"))\n        self.buffer_path = Path(os.environ.get("EVO1_QVLA_BUFFER_PATH", "/content/drive/MyDrive/Evo-1-results/evo1_torchao_w8a16_calib_buffer.jsonl"))\n        self.quant_scope = os.environ.get("EVO1_QVLA_QUANT_SCOPE", "both").strip().lower()\n        self.target_info = None\n        self.eval_apply_printed = False\n\n    def write_scales(self):\n        self.scales = self.stats.finalize()\n        self.scales["calibration_requests"] = int(min(self.request_i, self.calib_requests_target))\n        self.scales["calibration_target"] = int(self.calib_requests_target)\n        ti = self.target_info or {}\n        self.scales["quantized_scope"] = {\n            "backend": "torchao.Int8WeightOnlyConfig",\n            "quant_scope": self.quant_scope,\n            "language_model_linear": int(ti.get("language_model_linear", 0)),\n            "action_head_linear": int(ti.get("action_head_linear", 0)),\n            "target_total": int(ti.get("target_total", 0)),\n            "vision_model": 0,\n            "projector_mlp1": 0,\n        }\n        self.scales["target_names_sorted"] = sorted(ti.get("target_names", []))\n        self.scales["eval_apply_flags"] = {\n            "context_gain": os.environ.get("EVO1_QVLA_APPLY_CONTEXT_GAIN", "1"),\n            "atm": os.environ.get("EVO1_QVLA_APPLY_ATM", "1"),\n            "ohb_attn": os.environ.get("EVO1_QVLA_APPLY_OHB_ATTN", "1"),\n            "ohb_ff": os.environ.get("EVO1_QVLA_APPLY_OHB_FF", "1"),\n        }\n        self.scales["torchao"] = {\n            "backend": "torchao.quantization.Int8WeightOnlyConfig",\n            "wbits": 8,\n            "activation_quantization": "none",\n            "activations": "unchanged bf16/fp16 autocast",\n            "granularity": "PerRow/default",\n            "version": 2,\n        }\n        self.scales["calibration_buffer_path"] = str(self.buffer_path)\n        self.scales_path.parent.mkdir(parents=True, exist_ok=True)\n        self.scales_path.write_text(json.dumps(self.scales, indent=2))\n        meta_path = self.scales_path.with_name(self.scales_path.stem + "_meta.json")\n        meta = {\n            "checkpoint_dir": os.environ.get("EVO1_CKPT_DIR", ""),\n            "scope": self.scales["quantized_scope"],\n            "target_names_sorted": self.scales["target_names_sorted"],\n            "torchao": self.scales["torchao"],\n            "calibration_requests_target": self.calib_requests_target,\n            "calibration_requests_collected": self.scales.get("calibration_requests"),\n            "buffer_path": str(self.buffer_path),\n            "scales_path": str(self.scales_path),\n            "timestamp": time.time(),\n        }\n        meta_path.write_text(json.dumps(meta, indent=2))\n        print("[EVO1-W8A16] W8A16-CALIB-DONE", flush=True)\n        print("[EVO1-W8A16] CALIB-DONE", flush=True)  # backwards-compatible marker\n        print(f"[EVO1-W8A16] SCALES-WRITTEN: {self.scales_path}", flush=True)\n        print(f"[EVO1-W8A16] META-WRITTEN: {meta_path}", flush=True)\n        print(f"[EVO1-W8A16] CALIB_COLLECTED: {self.scales.get(\'calibration_requests\')}/{self.calib_requests_target}", flush=True)\n        for key in [\n            "context_cosine_mean", "context_rms_ratio_before", "context_gain_scalar",\n            "context_rms_ratio_after_scalar", "context_rms_ratio_after_vec_mean",\n            "atm_logit_ratio_before_mean", "atm_logit_ratio_after_mean",\n            "ohb_attn_ratio_before_mean", "ohb_attn_ratio_after_mean",\n            "ohb_ff_ratio_before_mean", "ohb_ff_ratio_after_mean",\n        ]:\n            print(f"[EVO1-W8A16] {key}: {self.scales.get(key)}", flush=True)\n        print(f"[EVO1-W8A16] ATM first layer alpha: {self.scales[\'atm_alpha_per_layer_head\'][0]}", flush=True)\n        print(f"[EVO1-W8A16] OHB attn beta: {self.scales[\'ohb_beta_attn_per_layer\']}", flush=True)\n        print(f"[EVO1-W8A16] OHB ff beta: {self.scales[\'ohb_beta_ff_per_layer\']}", flush=True)\n        install_qvla_action_patch(self.student, QVLAApply(scales=self.scales, role="student", collect=False, stats=None))\n        self.calib_done = True\n        try:\n            del self.teacher\n            self.teacher = None\n            gc.collect(); torch.cuda.empty_cache()\n            print("[EVO1-W8A16] FP16 teacher freed after calibration", flush=True)\n        except Exception as e:\n            print("[EVO1-W8A16] teacher free warning", repr(e), flush=True)\n\nasync def handle_request(websocket, state: QVLAServerState):\n    print("Client connected", flush=True)\n    try:\n        async for message in websocket:\n            data = json.loads(message)\n            state.request_i += 1\n            with open(state.buffer_path, "a") as bf:\n                if not state.calib_done and state.request_i <= state.calib_requests_target:\n                    bf.write(json.dumps({"request_index": state.request_i, "data": data}) + "\\n")\n            if not state.calib_done:\n                images, norm_state, prompt, image_mask, action_mask = prepare_inputs(data, state.normalizer)\n                seed = 123450 + state.request_i\n                with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):\n                    fused_fp = state.teacher.get_vl_embeddings(images=images, image_mask=image_mask, prompt=prompt, return_cls_only=None)\n                    fused_q = state.student.get_vl_embeddings(images=images, image_mask=image_mask, prompt=prompt, return_cls_only=None)\n                    state.stats.record_context(fused_fp, fused_q)\n                    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)\n                    action_teacher = state.teacher.predict_action(fused_fp, norm_state, action_mask=action_mask)\n                    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)\n                    _ = state.student.predict_action(fused_q, norm_state, action_mask=action_mask)\n                if state.request_i >= state.calib_requests_target:\n                    state.write_scales()\n                actions = action_to_json(action_teacher, state.normalizer)\n                await websocket.send(json.dumps(actions))\n                print(f"[EVO1-W8A16] CALIB request {state.request_i}/{state.calib_requests_target}", flush=True)\n            else:\n                if not state.eval_apply_printed:\n                    print("[EVO1-W8A16] SCALES_APPLIED_DURING_EVAL: True", flush=True)\n                    print(f"[EVO1-W8A16] APPLY_CONTEXT_GAIN: {os.environ.get(\'EVO1_QVLA_APPLY_CONTEXT_GAIN\',\'1\')}", flush=True)\n                    print(f"[EVO1-W8A16] APPLY_ATM: {os.environ.get(\'EVO1_QVLA_APPLY_ATM\',\'1\')}  # default ON: QuantVLA-style ATM; set 0 only for ablation", flush=True)\n                    print(f"[EVO1-W8A16] APPLY_OHB_ATTN: {os.environ.get(\'EVO1_QVLA_APPLY_OHB_ATTN\',\'1\')}", flush=True)\n                    print(f"[EVO1-W8A16] APPLY_OHB_FF: {os.environ.get(\'EVO1_QVLA_APPLY_OHB_FF\',\'1\')}", flush=True)\n                    state.eval_apply_printed = True\n                images, norm_state, prompt, image_mask, action_mask = prepare_inputs(data, state.normalizer)\n                with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):\n                    fused_q = state.student.get_vl_embeddings(images=images, image_mask=image_mask, prompt=prompt, return_cls_only=None)\n                    fused_q = QVLAApply(scales=state.scales).correct_context(fused_q)\n                    action = state.student.predict_action(fused_q, norm_state, action_mask=action_mask)\n                actions = action_to_json(action, state.normalizer)\n                await websocket.send(json.dumps(actions))\n                print(f"[EVO1-W8A16] EVAL request {state.request_i}", flush=True)\n    except websockets.exceptions.ConnectionClosed:\n        print("Client disconnected.", flush=True)\n\nif __name__ == "__main__":\n    ckpt_dir = os.environ.get("EVO1_CKPT_DIR", "/content/Evo1_LIBERO")\n    port = int(os.environ.get("EVO1_PORT", "9010"))\n    state = QVLAServerState()\n    state.buffer_path.parent.mkdir(parents=True, exist_ok=True)\n    force_recalib = os.environ.get("EVO1_QVLA_FORCE_RECALIB", "0") in ("1", "true", "True")\n    found_existing_scales = state.scales_path.exists()\n    print(f"[EVO1-W8A16] FOUND_EXISTING_SCALES: {found_existing_scales}", flush=True)\n    print(f"[EVO1-W8A16] FORCE_RECALIB: {force_recalib}", flush=True)\n    print(f"[EVO1-W8A16] torch: {torch.__version__}", flush=True)\n    print(f"[EVO1-W8A16] torchao: {getattr(torchao, \'__version__\', \'unknown\')}", flush=True)\n\n    if found_existing_scales and not force_recalib:\n        print("[EVO1-W8A16] Loading torchao W8A16 student for eval with existing scales...", flush=True)\n        state.student, state.normalizer = load_model_and_normalizer(ckpt_dir)\n        state.target_info = apply_torchao_w8a16_to_student(state.student)\n        state.scales = json.loads(state.scales_path.read_text())\n        scope = state.scales.get("quantized_scope", {})\n        expected_target_names = sorted(state.target_info.get("target_names", []))\n        scale_target_names = sorted(state.scales.get("target_names_sorted", []))\n        scope_match = (\n            scope.get("backend") == "torchao.Int8WeightOnlyConfig"\n            and scope.get("quant_scope") == state.quant_scope\n            and int(scope.get("target_total", -1)) == int(state.target_info.get("target_total", -2))\n        )\n        target_set_match = scale_target_names == expected_target_names\n        torchao_match = state.scales.get("torchao", {}).get("backend") == "torchao.quantization.Int8WeightOnlyConfig"\n        formula_match = state.scales.get("formula_version") == "evo1_torchao_w8a16_diag_v1"\n        calib_match = int(state.scales.get("calibration_requests", 0)) >= state.calib_requests_target\n        config_match = scope_match and target_set_match and torchao_match and formula_match and calib_match\n        print(f"[EVO1-W8A16] SCALES_SCOPE_MATCH: {scope_match}", flush=True)\n        print(f"[EVO1-W8A16] SCALES_TARGET_SET_MATCH: {target_set_match}", flush=True)\n        print(f"[EVO1-W8A16] SCALES_TORCHAO_MATCH: {torchao_match}", flush=True)\n        print(f"[EVO1-W8A16] SCALES_FORMULA_MATCH: {formula_match}", flush=True)\n        print(f"[EVO1-W8A16] SCALES_CALIB_TARGET_MATCH: {calib_match}", flush=True)\n        print(f"[EVO1-W8A16] SCALES_CONFIG_MATCH: {config_match}", flush=True)\n        if not config_match:\n            raise RuntimeError("Existing scales config does not exactly match current torchao W8A16 target scope/settings/formula/calibration target; rerun with FORCE_RECALIB=1")\n        install_qvla_action_patch(state.student, QVLAApply(scales=state.scales, role="student", collect=False, stats=None))\n        state.calib_done = True\n        print("[EVO1-W8A16] CALIBRATION_SKIPPED: True", flush=True)\n        print("[EVO1-W8A16] CALLS-INSERTED-AFTER-LOAD", flush=True)\n        print(f"[EVO1-W8A16] SCALES-LOADED: {state.scales_path}", flush=True)\n    else:\n        if state.buffer_path.exists(): state.buffer_path.unlink()\n        print("[EVO1-W8A16] Loading FP16 teacher...", flush=True)\n        state.teacher, state.normalizer = load_model_and_normalizer(ckpt_dir)\n        print("[EVO1-W8A16] Loading torchao W8A16 student...", flush=True)\n        state.student, _ = load_model_and_normalizer(ckpt_dir)\n        print("[EVO1-W8A16] Applying torchao Int8WeightOnlyConfig to selected student linears...", flush=True)\n        state.target_info = apply_torchao_w8a16_to_student(state.student)\n        state.stats = QVLAStats(num_layers=len(state.student.action_head.transformer_blocks), num_heads=state.student.action_head.transformer_blocks[0].attn.num_heads)\n        install_qvla_action_patch(state.teacher, QVLAApply(role="teacher", collect=True, stats=state.stats))\n        install_qvla_action_patch(state.student, QVLAApply(role="student", collect=True, stats=state.stats))\n        print("[EVO1-W8A16] CALIBRATION_SKIPPED: False", flush=True)\n        print("[EVO1-W8A16] CALLS-INSERTED-AFTER-LOAD", flush=True)\n        print("[EVO1-W8A16] Ready for fixed calibration buffer collection", flush=True)\n\n    async def main():\n        print(f"EVO_1 torchao W8A16 server running at ws://0.0.0.0:{port}", flush=True)\n        async with websockets.serve(lambda ws: handle_request(ws, state), "0.0.0.0", port, max_size=100_000_000, ping_interval=None, ping_timeout=None, close_timeout=30):\n            await asyncio.Future()\n    asyncio.run(main())\n')
print("WROTE SERVER SCRIPT:", SERVER_SCRIPT)
print("SERVER SCRIPT BYTES:", SERVER_SCRIPT.stat().st_size)


WROTE SERVER SCRIPT: /content/drive/MyDrive/Evo-1/Evo_1/scripts/Evo1_server_torchao_w8a16_action_only_atm_ohb.py
SERVER SCRIPT BYTES: 36396


In [16]:

# QVLA3b. Static regex sanity check against known Evo-1 runtime names before starting server.
import re
llm_all_pat = re.compile(
    r"^embedder\.model\.language_model\.(?:model\.layers|layers)\.\d+\."
    r"(?:self_attn\.(?:q_proj|k_proj|v_proj|o_proj)|mlp\.(?:gate_proj|up_proj|down_proj))$"
)
llm_attn_name = "embedder.model.language_model.model.layers.0.self_attn.q_proj"
llm_mlp_name = "embedder.model.language_model.model.layers.0.mlp.gate_proj"
vision_name = "embedder.model.vision_model.encoder.layers.0.attn.qkv"
action_ffn_name = "action_head.transformer_blocks.0.ff.0"
action_attn_name = "action_head.transformer_blocks.0.attn.out_proj"
action_pat = re.compile(r"^action_head\.transformer_blocks\.\d+\..*$")
print("REGEX_TEST LLM_ATTN should match:", bool(llm_all_pat.match(llm_attn_name)), llm_attn_name)
print("REGEX_TEST LLM_MLP should match:", bool(llm_all_pat.match(llm_mlp_name)), llm_mlp_name)
print("REGEX_TEST VISION should NOT match:", bool(llm_all_pat.match(vision_name)), vision_name)
print("REGEX_TEST ACTION_FFN should match:", bool(action_pat.match(action_ffn_name)), action_ffn_name)
print("REGEX_TEST ACTION_ATTN should match if Linear exists:", bool(action_pat.match(action_attn_name)), action_attn_name)
assert llm_all_pat.match(llm_attn_name)
assert llm_all_pat.match(llm_mlp_name)
assert not llm_all_pat.match(vision_name)
assert action_pat.match(action_ffn_name)
assert action_pat.match(action_attn_name)
print("REGEX_TEST_PASS: LLM all-linear scope + action-head transformer-block Linear scope")


REGEX_TEST LLM_ATTN should match: True embedder.model.language_model.model.layers.0.self_attn.q_proj
REGEX_TEST LLM_MLP should match: True embedder.model.language_model.model.layers.0.mlp.gate_proj
REGEX_TEST VISION should NOT match: False embedder.model.vision_model.encoder.layers.0.attn.qkv
REGEX_TEST ACTION_FFN should match: True action_head.transformer_blocks.0.ff.0
REGEX_TEST ACTION_ATTN should match if Linear exists: True action_head.transformer_blocks.0.attn.out_proj
REGEX_TEST_PASS: LLM all-linear scope + action-head transformer-block Linear scope


### Server start and proof lines

This starts the torchao W8A16 server and refuses to continue unless torchao proof lines appear.

In [17]:

# QVLA4. Start one server in calibration-then-eval mode.
# It loads FP16 teacher + torchao W8A16 student, then waits for fixed calibration observations.
import subprocess, os, time, re, json
from pathlib import Path

kill_port_9010()
if SERVER_LOG.exists(): SERVER_LOG.unlink()
if FORCE_RECALIB and SCALES_PATH.exists():
    print("FORCE_RECALIB=True, removing old scales:", SCALES_PATH)
    SCALES_PATH.unlink()
if FORCE_RECALIB and SCALES_META_PATH.exists():
    SCALES_META_PATH.unlink()
if FORCE_RECALIB and DIAG_PATH.exists():
    DIAG_PATH.unlink()
if FORCE_RECALIB and BUFFER_PATH.exists():
    BUFFER_PATH.unlink()

env = {
    **os.environ,
    "MAMBA_ROOT_PREFIX": MAMBA_ROOT,
    "EVO1_CKPT_DIR": str(LOCAL_CKPT),
    "EVO1_PORT": str(PORT),
    "EVO1_QVLA_MODE": "calib_then_eval",
    "EVO1_QVLA_CALIB_REQUESTS": str(CALIB_REQUESTS),
    "EVO1_QVLA_SCALES_PATH": str(SCALES_PATH),
    "EVO1_QVLA_BUFFER_PATH": str(BUFFER_PATH),
    "EVO1_QVLA_QUANT_SCOPE": QUANT_SCOPE,
    "EVO1_QVLA_APPLY_CONTEXT_GAIN": APPLY_CONTEXT_GAIN,
    "EVO1_QVLA_APPLY_ATM": APPLY_ATM,
    "EVO1_QVLA_APPLY_OHB_ATTN": APPLY_OHB_ATTN,
    "EVO1_QVLA_APPLY_OHB_FF": APPLY_OHB_FF,
    "EVO1_QVLA_FORCE_RECALIB": "1" if FORCE_RECALIB else "0",
    "TORCH_COMPILE_DISABLE": "1",
    "TORCHDYNAMO_DISABLE": "1",
    "TORCH_CUDA_GRAPH_DISABLE": "1",
}

cmd = [MAMBA, "run", "-n", "Evo1", "python", "-u", str(SERVER_SCRIPT)]
logf = open(SERVER_LOG, "w")
p = subprocess.Popen(cmd, cwd=str(SCRIPTS), env=env, stdout=logf, stderr=subprocess.STDOUT, text=True)
print("torchao W8A16 server pid:", p.pid)
print("torchao W8A16 server log:", SERVER_LOG)
print("QUANT_SCOPE:", QUANT_SCOPE)

deadline = time.time() + 900
required_any = ["EVO_1 torchao W8A16 server running", "Ready for fixed calibration buffer collection", "CALIBRATION_SKIPPED: True"]
required_proof = [
    "[EVO1-W8A16] QUANT_BACKEND: torchao.Int8WeightOnlyConfig",
    "[EVO1-W8A16] TORCHAO-INT8-WEIGHTONLY-PROOF: True",
]
while time.time() < deadline:
    time.sleep(5)
    txt = SERVER_LOG.read_text(errors="ignore") if SERVER_LOG.exists() else ""
    if "Traceback (most recent call last)" in txt or "[FATAL]" in txt or "RuntimeError:" in txt:
        print("SERVER LOG FAILURE MARKER FOUND")
        print(txt[-8000:])
        raise RuntimeError("Server failed during startup")
    if all(x in txt for x in required_proof) and any(x in txt for x in required_any):
        print("SERVER STARTED AND PROOF LINES FOUND")
        print(txt[-6000:])
        break
else:
    txt = SERVER_LOG.read_text(errors="ignore") if SERVER_LOG.exists() else ""
    print(txt[-8000:])
    raise RuntimeError("Timed out waiting for torchao W8A16 server proof lines")

r = subprocess.run("ss -ltnp | grep ':9010'", shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(r.stdout)
if r.returncode != 0:
    raise RuntimeError("Server proof passed but port 9010 is not listening")


+ ss -ltnp | grep ':9010' || true
no process currently listening on 9010
torchao W8A16 server pid: 43388
torchao W8A16 server log: /content/evo1_torchao_w8a16_action_only_atm_ohb_server.log
QUANT_SCOPE: action_only
SERVER STARTED AND PROOF LINES FOUND
le. If you want to force a new download, use `force_download=True`.
  warnings.warn(
num_inference_timesteps 32
[EVO1-W8A16] MODEL_DTYPE_AFTER_LOAD: bfloat16
[EVO1-W8A16] Loading torchao W8A16 student...
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
num_inference_timesteps 32
[EVO1-W8A16] MODEL_DTYPE_AFTER_LOAD: bfloat16
[EVO1-W8A16] Applying torchao Int8WeightOnlyConfig to selected student linears...
[EVO1-W8A16] QUANT_BACKEND: torchao.Int8WeightOnlyConfig
[EVO1-W8A16] QUANT_SCOPE: action_only
[EVO1-W8A16] preflight MATCH-LLM-ALL-LINEAR: 98
[EVO1-W8A16] preflight MATCH-ACTION-BLOCK-LINEAR: 24
[EVO1-W8A16] SELECTED-TARGETS: 24
[EVO1-W8A16] EXPECTED-LLM-ALL-LINEAR: 98


In [18]:

# QVLA4b. Extra sanity check: prove torchao backend, not a local Linear wrapper.
txt = SERVER_LOG.read_text(errors="ignore") if SERVER_LOG.exists() else ""
required = [
    "[EVO1-W8A16] QUANT_BACKEND: torchao.Int8WeightOnlyConfig",
    "[EVO1-W8A16] TORCHAO-INT8-WEIGHTONLY-PROOF: True",
    "[EVO1-W8A16] EXPECTED-LLM-ALL-LINEAR: 98",
]
missing = [x for x in required if x not in txt]
if missing:
    raise RuntimeError("Missing torchao proof lines: " + repr(missing))
for banned in ["W8A16" + "Linear", "DuQuant" + "Linear"]:
    if banned in txt:
        raise RuntimeError(f"Banned homemade/outdated backend token appeared in server log: {banned}")
print("torchao W8A16 backend verified from server log.")


torchao W8A16 backend verified from server log.


## STOP / EXPENSIVE CELL: QVLA5 calibration episode collection

Run this cell only when you are ready to collect calibration samples.  
It is **not measured eval**. It runs fixed LIBERO episode IDs until `CALIB_REQUESTS=32` is reached, then saves scales to Drive.

After this cell, check that it prints:
- `CALIBRATION READY / SCALES AVAILABLE`
- `CALIBRATION_DIAGNOSTICS_PASS: True`
- before/after drift ratios moving closer to 1.0

You may stop after QVLA6 if you only want calibration and diagnostics.


In [19]:

# QVLA5. Run fixed unlabeled calibration buffer through the same baseline LIBERO client.
# This is not measured eval. The server responds with FP16 teacher actions and compares teacher/student internally.
import subprocess, os, json, time, re
from pathlib import Path

CALIB_RESULTS = RESULTS / "calibration_logs"
CALIB_RESULTS.mkdir(parents=True, exist_ok=True)

def server_is_listening():
    r = subprocess.run("ss -ltnp | grep ':9010'", shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return r.returncode == 0

def run_calib_episode(suite, task_id, ep):
    ckpt_name = f"{CALIB_TAG}_{suite}_task{task_id}_ep{ep}"
    log_path = CALIB_RESULTS / f"{ckpt_name}.log"
    if not server_is_listening():
        raise RuntimeError("Server is not listening on 9010 before calibration episode")
    env = {
        **os.environ,
        "MAMBA_ROOT_PREFIX": MAMBA_ROOT,
        "SINGLE_SUITE": suite,
        "SINGLE_TASK_ID": str(task_id),
        "SINGLE_EP_INDEX": str(ep),
        "SINGLE_MAX_STEPS": str(MAX_STEPS_BY_SUITE[suite]),
        "SINGLE_CKPT_NAME": ckpt_name,
    }
    cmd = [MAMBA, "run", "-n", "libero", "python", "-u", "libero_client_single_episode_runtime.py"]
    print("\nCALIB RUN", ckpt_name)
    with open(log_path, "w") as f:
        p = subprocess.run(cmd, cwd=str(LIBERO_EVAL), env=env, stdout=f, stderr=subprocess.STDOUT, text=True)
    txt = log_path.read_text(errors="ignore")
    print("calib returncode", p.returncode)
    print("--- calib client tail ---")
    print("\n".join(txt.splitlines()[-25:]))

if SCALES_PATH.exists() and not FORCE_RECALIB:
    print("FOUND EXISTING SCALES, calibration skipped:", SCALES_PATH)
else:
    for ep in CALIB_EPISODES:
        if SCALES_PATH.exists():
            print("scales written; stop calibration episodes")
            break
        run_calib_episode("libero_spatial", 0, ep)
        # Give server time to flush scale JSON.
        time.sleep(2)
        txt = SERVER_LOG.read_text(errors="ignore") if SERVER_LOG.exists() else ""
        m = re.findall(r"CALIB request (\d+)/(\d+)", txt)
        if m:
            print("CALIB_PROGRESS:", m[-1][0] + "/" + m[-1][1])

# Wait for server to finalize calibration or load existing scales.
for i in range(90):
    txt = SERVER_LOG.read_text(errors="ignore") if SERVER_LOG.exists() else ""
    if SCALES_PATH.exists() and ("[EVO1-W8A16] W8A16-CALIB-DONE" in txt or "[EVO1-W8A16] CALIB-DONE" in txt or "[EVO1-W8A16] CALIBRATION_SKIPPED: True" in txt):
        print("CALIBRATION READY / SCALES AVAILABLE")
        print("scales:", SCALES_PATH)
        print("buffer:", BUFFER_PATH)
        break
    if i % 5 == 0:
        print("waiting for calibration/scales...", "scales exists", SCALES_PATH.exists())
        print("--- server tail ---")
        print("\n".join(txt.splitlines()[-80:]))
    time.sleep(5)
else:
    raise RuntimeError("Calibration did not finish. It must reach CALIB_REQUESTS exactly; do not eval without scales.")

scales = json.loads(SCALES_PATH.read_text())
diag_keys = [
    "context_cosine_mean",
    "context_rms_ratio_before", "context_gain_scalar", "context_rms_ratio_after_scalar", "context_rms_ratio_after_vec_mean",
    "atm_logit_ratio_before_mean", "atm_logit_ratio_after_mean",
    "ohb_attn_ratio_before_mean", "ohb_attn_ratio_after_mean",
    "ohb_ff_ratio_before_mean", "ohb_ff_ratio_after_mean",
    "calibration_requests", "quantized_scope", "torchao", "eval_apply_flags", "formulas"
]
print(json.dumps({k: scales.get(k) for k in diag_keys}, indent=2))
# hard sanity gates before eval
if int(scales.get("calibration_requests", 0)) < CALIB_REQUESTS:
    raise RuntimeError("Calibration requests below target; do not eval")
scope = scales.get("quantized_scope", {})
if scope.get("backend") != "torchao.Int8WeightOnlyConfig":
    raise RuntimeError(f"Wrong backend in scales: {scope}")
if scope.get("quant_scope") != QUANT_SCOPE:
    raise RuntimeError(f"Wrong quant scope in scales: {scope.get('quant_scope')} != {QUANT_SCOPE}")
if QUANT_SCOPE in ("both", "llm_only") and scope.get("language_model_linear") != 98:
    raise RuntimeError(f"Wrong LLM all-linear target count in scales; expected 98, got {scope}")
if QUANT_SCOPE in ("both", "action_only") and int(scope.get("action_head_linear", 0)) < 16:
    raise RuntimeError(f"Wrong action-head Linear target count in scales; expected at least 16, got {scope}")
for k in ["context_rms_ratio_after_scalar", "atm_logit_ratio_after_mean", "ohb_attn_ratio_after_mean"]:
    v = float(scales.get(k, 999))
    if not (0.25 <= v <= 4.0):
        raise RuntimeError(f"Suspicious calibration diagnostic {k}={v}")

def closer_to_one(before_key, after_key, tolerance=0.05):
    before = float(scales.get(before_key, 999))
    after = float(scales.get(after_key, 999))
    print(f"DRIFT_CHECK {before_key}->{after_key}: before={before:.6f} after={after:.6f} |before-1|={abs(before-1):.6f} |after-1|={abs(after-1):.6f}")
    if abs(after - 1.0) > abs(before - 1.0) + tolerance:
        raise RuntimeError(f"Correction made drift worse: {before_key}={before}, {after_key}={after}")

closer_to_one("context_rms_ratio_before", "context_rms_ratio_after_scalar")
closer_to_one("atm_logit_ratio_before_mean", "atm_logit_ratio_after_mean")
closer_to_one("ohb_attn_ratio_before_mean", "ohb_attn_ratio_after_mean")

# Save separate diagnostics JSON for future skip/debug.
DIAG_KEYS_TO_SAVE = [
    "calibration_requests", "calibration_target", "context_cosine_mean",
    "context_rms_ratio_before", "context_rms_ratio_after_scalar", "context_rms_ratio_after_vec_mean",
    "atm_logit_ratio_before_mean", "atm_logit_ratio_after_mean",
    "ohb_attn_ratio_before_mean", "ohb_attn_ratio_after_mean",
    "ohb_ff_ratio_before_mean", "ohb_ff_ratio_after_mean",
    "eval_apply_flags", "quantized_scope", "torchao", "formulas",
]
DIAG_PATH.write_text(json.dumps({k: scales.get(k) for k in DIAG_KEYS_TO_SAVE}, indent=2))
print("CALIB_DIAGNOSTICS_WRITTEN:", DIAG_PATH)

print("CALIBRATION_DIAGNOSTICS_PASS: True")



CALIB RUN torchao_w8a16_action_only_calib_fixed_spatial_task0_libero_spatial_task0_ep0
calib returncode 0
--- calib client tail ---
	err = EGL_NOT_INITIALIZED,
	baseOperation = eglDestroyContext,
	cArguments = (
	),
	result = 0
)
Exception ignored in: <function EGLGLContext.__del__ at 0x7f3d2749b550>
Traceback (most recent call last):
  File "/content/micromamba-root/envs/libero/lib/python3.8/site-packages/robosuite/renderers/context/egl_context.py", line 155, in __del__
    self.free()
  File "/content/micromamba-root/envs/libero/lib/python3.8/site-packages/robosuite/renderers/context/egl_context.py", line 150, in free
    EGL.eglDestroyContext(EGL_DISPLAY, self._context)
  File "/content/micromamba-root/envs/libero/lib/python3.8/site-packages/OpenGL/error.py", line 230, in glCheckError
    raise self._errorClass(
OpenGL.raw.EGL._errors.EGLError: EGLError(
	err = EGL_NOT_INITIALIZED,
	baseOperation = eglDestroyContext,
	cArguments = (
	),
	result = 0
)
CALIB_PROGRESS: 6/32

CALIB RUN

In [20]:

# QVLA5b. Persist calibration metadata and copy server log to Drive.
# This cell is cheap. Run it after QVLA5 so debugging survives Colab restart.
from pathlib import Path
import json, shutil, time, os

assert SCALES_PATH.exists(), f"Missing scales file: {SCALES_PATH}"
scales = json.loads(SCALES_PATH.read_text())

meta = {
    "notebook": "evo1_FP16base_torchao_W8A16_alllinear_actionlinear",
    "timestamp": time.time(),
    "checkpoint_drive_path": str(DRIVE_CKPT),
    "checkpoint_local_path": str(LOCAL_CKPT),
    "repo_path": str(REPO),
    "quant_scope_expected": scales.get("quantized_scope", {}),
    "target_names_sorted": scales.get("target_names_sorted", []),
    "torchao_expected": {
        "backend": "torchao.quantization.Int8WeightOnlyConfig",
        "wbits": 8,
        "activation_quantization": "none",
    },
    "calibration_expected": {
        "calib_requests_target": CALIB_REQUESTS,
        "calib_episode_ids_fixed": CALIB_EPISODES,
        "buffer_path": str(BUFFER_PATH),
        "scales_path": str(SCALES_PATH),
        "diagnostics_path": str(DIAG_PATH),
    },
    "calibration_observed": {
        "calibration_requests": scales.get("calibration_requests"),
        "calibration_target": scales.get("calibration_target"),
        "context_cosine_mean": scales.get("context_cosine_mean"),
        "context_rms_ratio_before": scales.get("context_rms_ratio_before"),
        "context_rms_ratio_after_scalar": scales.get("context_rms_ratio_after_scalar"),
        "atm_logit_ratio_before_mean": scales.get("atm_logit_ratio_before_mean"),
        "atm_logit_ratio_after_mean": scales.get("atm_logit_ratio_after_mean"),
        "ohb_attn_ratio_before_mean": scales.get("ohb_attn_ratio_before_mean"),
        "ohb_attn_ratio_after_mean": scales.get("ohb_attn_ratio_after_mean"),
    },
    "formula_reminder": {
        "context_gain": "rms(context_fp) / rms(context_q)",
        "ATM_alpha": "std(attention_logits_fp) / std(attention_logits_q)",
        "OHB_beta": "rms(output_fp) / rms(output_q)",
        "ratio_before": "quant_stat / fp_stat",
        "ratio_after": "corrected_quant_stat / fp_stat; should be closer to 1.0",
    }
}

SCALES_META_PATH.write_text(json.dumps(meta, indent=2))
print("SCALES_META_WRITTEN:", SCALES_META_PATH)
print("CALIB_DIAGNOSTICS_PATH:", DIAG_PATH, "exists=", DIAG_PATH.exists())

if SERVER_LOG.exists():
    log_copy = SCALES_DIR / "server_log_after_calibration.txt"
    shutil.copy2(SERVER_LOG, log_copy)
    print("SERVER_LOG_COPIED_TO_DRIVE:", log_copy)
else:
    print("SERVER_LOG_MISSING_IN_CONTENT:", SERVER_LOG)

print("FOUND EXISTING SCALES:", SCALES_PATH.exists())
print("CONFIG MATCH SHOULD BE EXACTLY torchao W8A16, current QUANT_SCOPE, sorted target-name set, calibration target")
print("CALIBRATION SAVED AND SAFE TO REUSE IF THESE SETTINGS DO NOT CHANGE.")


SCALES_META_WRITTEN: /content/drive/MyDrive/Evo-1-results/quantvla_scales/evo1_torchao_w8a16_action_only_atm_ohb_v1/scales_meta.json
CALIB_DIAGNOSTICS_PATH: /content/drive/MyDrive/Evo-1-results/quantvla_scales/evo1_torchao_w8a16_action_only_atm_ohb_v1/calib_diagnostics.json exists= True
SERVER_LOG_COPIED_TO_DRIVE: /content/drive/MyDrive/Evo-1-results/quantvla_scales/evo1_torchao_w8a16_action_only_atm_ohb_v1/server_log_after_calibration.txt
FOUND EXISTING SCALES: True
CONFIG MATCH SHOULD BE EXACTLY torchao W8A16, current QUANT_SCOPE, sorted target-name set, calibration target
CALIBRATION SAVED AND SAFE TO REUSE IF THESE SETTINGS DO NOT CHANGE.


In [21]:

# QVLA6. Websocket-only check after calibration: same server should now be in eval mode.
import subprocess, os
ws_test_code = """
import asyncio, websockets
async def main():
    url = 'ws://127.0.0.1:9010'
    print('trying', url)
    async with websockets.connect(url, max_size=100_000_000, ping_interval=None, ping_timeout=None, close_timeout=30) as ws:
        print('CONNECTED OK')
asyncio.run(main())
"""
subprocess.run([MAMBA, "run", "-n", "libero", "python", "-c", ws_test_code], env={**os.environ, "MAMBA_ROOT_PREFIX": MAMBA_ROOT}, check=True)
print("Ready for measured eval. Scales loaded in running server:", SCALES_PATH.exists())


Ready for measured eval. Scales loaded in running server: True


## SAFE STOP POINT BEFORE MEASURED EVAL

If QVLA5 printed `CALIBRATION_DIAGNOSTICS_PASS: True`, calibration is saved in Drive and can be reused later.

Measured eval starts at QVLA8.  
Do **not** run QVLA8 until the diagnostics look sane.


In [22]:

# QVLA7. Measured evaluation settings and REQUIRED calibration diagnostics gate.
import json, math

print("Measured eval tag:", TAG)
print("Total subprocess runs:", len(TASK_SUITES) * len(TASK_IDS) * len(EPISODES))
print("Scope: torchao W8A16 static weight-only; QUANT_SCOPE =", QUANT_SCOPE)
print("Scales path:", SCALES_PATH)
print("Meta path:", SCALES_META_PATH)
print("Diagnostics path:", DIAG_PATH)

if not SCALES_PATH.exists():
    raise RuntimeError("Missing scales.json. Run QVLA5 before eval.")
if not SCALES_META_PATH.exists():
    raise RuntimeError("Missing scales_meta.json. Run QVLA5b before eval.")
if not DIAG_PATH.exists():
    raise RuntimeError("Missing calib_diagnostics.json. Run QVLA5/QVLA5b before eval.")

scales = json.loads(SCALES_PATH.read_text())
meta = json.loads(SCALES_META_PATH.read_text())
diag = json.loads(DIAG_PATH.read_text())
required_diag = [
    "calibration_requests", "calibration_target", "context_cosine_mean",
    "context_rms_ratio_before", "context_rms_ratio_after_scalar",
    "atm_logit_ratio_before_mean", "atm_logit_ratio_after_mean",
    "ohb_attn_ratio_before_mean", "ohb_attn_ratio_after_mean",
    "ohb_ff_ratio_before_mean", "ohb_ff_ratio_after_mean",
    "eval_apply_flags", "quantized_scope", "torchao", "target_names_sorted",
]
missing = [k for k in required_diag if k not in diag or diag.get(k) is None]
if missing:
    raise RuntimeError("Missing calibration diagnostic keys: " + repr(missing))

if int(diag.get("calibration_requests", 0)) < CALIB_REQUESTS:
    raise RuntimeError(f"Calibration target not reached: {diag.get('calibration_requests')} < {CALIB_REQUESTS}")
scope = diag.get("quantized_scope", {})
if scope.get("backend") != "torchao.Int8WeightOnlyConfig":
    raise RuntimeError(f"Wrong backend in diagnostics: {scope}")
if scope.get("quant_scope") != QUANT_SCOPE:
    raise RuntimeError(f"Wrong quant scope in diagnostics: {scope.get('quant_scope')} != {QUANT_SCOPE}")
if QUANT_SCOPE in ("both", "llm_only") and scope.get("language_model_linear") != 98:
    raise RuntimeError(f"LLM all-linear count wrong: {scope}")
if QUANT_SCOPE in ("both", "action_only") and int(scope.get("action_head_linear", 0)) < 16:
    raise RuntimeError(f"Action-head linear count too small: {scope}")
if diag.get("torchao", {}).get("backend") != "torchao.quantization.Int8WeightOnlyConfig":
    raise RuntimeError(f"Wrong torchao metadata: {diag.get('torchao')}")

for k in [
    "context_cosine_mean", "context_rms_ratio_before", "context_rms_ratio_after_scalar",
    "atm_logit_ratio_before_mean", "atm_logit_ratio_after_mean",
    "ohb_attn_ratio_before_mean", "ohb_attn_ratio_after_mean",
    "ohb_ff_ratio_before_mean", "ohb_ff_ratio_after_mean",
]:
    v = float(diag[k])
    if not math.isfinite(v):
        raise RuntimeError(f"Non-finite diagnostic {k}: {v}")
    print(k, "=", v)

print("QVLA7 DIAGNOSTIC GATE PASSED: torchao W8A16 scales/meta/diagnostics exist and calibration target was reached.")


Measured eval tag: torchao_w8a16_action_only_atm_ohb_spatial_task0_4ep
Total subprocess runs: 4
Scope: torchao W8A16 static weight-only; QUANT_SCOPE = action_only
Scales path: /content/drive/MyDrive/Evo-1-results/quantvla_scales/evo1_torchao_w8a16_action_only_atm_ohb_v1/scales.json
Meta path: /content/drive/MyDrive/Evo-1-results/quantvla_scales/evo1_torchao_w8a16_action_only_atm_ohb_v1/scales_meta.json
Diagnostics path: /content/drive/MyDrive/Evo-1-results/quantvla_scales/evo1_torchao_w8a16_action_only_atm_ohb_v1/calib_diagnostics.json


RuntimeError: Missing calibration diagnostic keys: ['target_names_sorted']

In [23]:

# QVLA8. Run measured eval one episode per subprocess, print SUCCESS/FAIL immediately.
import subprocess, os, time, json, re
from pathlib import Path

EVAL_RESULTS = RESULTS / "eval_logs"
EVAL_RESULTS.mkdir(parents=True, exist_ok=True)

# # Hard gate: QVLA8 must not run unless QVLA7 diagnostics passed/saved.
# if not (SCALES_PATH.exists() and SCALES_META_PATH.exists() and DIAG_PATH.exists()):
#     raise RuntimeError("Missing scales/meta/diagnostics. Run QVLA5, QVLA5b, and QVLA7 before eval.")
# _diag = json.loads(DIAG_PATH.read_text())
# if int(_diag.get("calibration_requests", 0)) < int(_diag.get("calibration_target", 999999)):
#     raise RuntimeError("Calibration incomplete in diagnostics; aborting measured eval.")
# if _diag.get("torchao", {}).get("backend") != "torchao.quantization.Int8WeightOnlyConfig":
#     raise RuntimeError("Diagnostics are not torchao Int8WeightOnlyConfig; aborting measured eval.")
# _scope = _diag.get("quantized_scope", {})
# if _scope.get("backend") != "torchao.Int8WeightOnlyConfig" or _scope.get("quant_scope") != QUANT_SCOPE:
#     raise RuntimeError(f"Wrong quantized scope for this notebook: {_scope}")
# if QUANT_SCOPE in ("both", "llm_only") and _scope.get("language_model_linear") != 98:
#     raise RuntimeError(f"Wrong LLM count: {_scope}")
# if QUANT_SCOPE in ("both", "action_only") and int(_scope.get("action_head_linear", 0)) < 16:
#     raise RuntimeError(f"Wrong action-head count: {_scope}")
# print("QVLA8 DIAGNOSTIC GATE PASSED: torchao W8A16 scales/meta/diagnostics exist and calibration target was reached.")

def server_is_listening():
    r = subprocess.run("ss -ltnp | grep ':9010'", shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return r.returncode == 0

def classify_log(text, returncode):
    rewards = re.findall(r"reward=([0-9.]+)", text)
    dones = re.findall(r"done=(True|False)", text)
    success = ("reward=1.00" in text) or ("done=True" in text) or ("✅ Success" in text)
    egl_cleanup = ("EGL_NOT_INITIALIZED" in text or "EGLGLContext.__del__" in text or "MjRenderContext.__del__" in text)
    websocket_crash = ("ConnectionClosedError" in text or "received 1011" in text or "internal error" in text)
    python_traceback = ("Traceback (most recent call last)" in text and not egl_cleanup)
    crashed = (returncode != 0) or websocket_crash or python_traceback
    return success, crashed, egl_cleanup, rewards[-10:], dones[-10:]

def run_one(suite, task_id, ep):
    ckpt_name = f"{TAG}_{suite}_task{task_id}_ep{ep}"
    log_path = EVAL_RESULTS / f"{ckpt_name}.log"
    done_marker = EVAL_RESULTS / f"{ckpt_name}.done.json"
    if done_marker.exists():
        rec = json.loads(done_marker.read_text())
        print("SKIP done:", ckpt_name, "SUCCESS" if rec.get("success_detected") else "FAIL")
        return rec
    if not server_is_listening():
        raise RuntimeError("Server is not listening on 9010. Do not evaluate.")
    if not SCALES_PATH.exists():
        raise RuntimeError("Missing calibration scales. Run QVLA5 before eval.")
    env = {
        **os.environ,
        "MAMBA_ROOT_PREFIX": MAMBA_ROOT,
        "SINGLE_SUITE": suite,
        "SINGLE_TASK_ID": str(task_id),
        "SINGLE_EP_INDEX": str(ep),
        "SINGLE_MAX_STEPS": str(MAX_STEPS_BY_SUITE[suite]),
        "SINGLE_CKPT_NAME": ckpt_name,
    }
    cmd = [MAMBA, "run", "-n", "libero", "python", "-u", "libero_client_single_episode_runtime.py"]
    print("\nRUN", ckpt_name)
    with open(log_path, "w") as f:
        p = subprocess.run(cmd, cwd=str(LIBERO_EVAL), env=env, stdout=f, stderr=subprocess.STDOUT, text=True)
    text = log_path.read_text(errors="ignore")
    success, crashed, egl_cleanup, last_rewards, last_dones = classify_log(text, p.returncode)
    rec = {
        "suite": suite, "task_id": task_id, "episode_index": ep,
        "returncode": p.returncode, "success_detected": bool(success), "crashed_detected": bool(crashed),
        "last_rewards": last_rewards, "last_dones": last_dones,
        "ignored_egl_cleanup_warning": bool(egl_cleanup),
        "log_path": str(log_path), "timestamp": time.time(),
        "scales_path": str(SCALES_PATH), "calib_buffer_path": str(BUFFER_PATH),
    }
    done_marker.write_text(json.dumps(rec, indent=2))
    print(("SUCCESS" if success else "FAIL"), "returncode=", p.returncode, "server_crash=", crashed, "egl_cleanup_warning=", egl_cleanup)
    print("last_rewards:", last_rewards)
    print("last_dones:", last_dones)
    print("--- tail ---")
    print("\n".join(text.splitlines()[-25:]))
    return rec

records = []
for suite in TASK_SUITES:
    for task_id in TASK_IDS:
        for ep in EPISODES:
            records.append(run_one(suite, task_id, ep))

succ = sum(r["success_detected"] for r in records)
server_crash = sum(r["crashed_detected"] for r in records)
task_fail = sum((not r["success_detected"]) and (not r["crashed_detected"]) for r in records)
egl_warn = sum(r.get("ignored_egl_cleanup_warning", False) for r in records)
print("\nMEASURED EVAL SUMMARY:")
print("eval_runs", len(records))
print("success", succ)
print("task_fail", task_fail)
print("server_crash", server_crash)
print("egl_cleanup_warning", egl_warn)
print("success_rate", succ / len(records) if records else None)
for r in records:
    label = "SUCCESS" if r["success_detected"] else ("SERVER_CRASH" if r["crashed_detected"] else "TASK_FAIL")
    print(f"ep{r['episode_index']}: {label} | returncode={r['returncode']} | last_reward={r['last_rewards'][-1] if r['last_rewards'] else 'NA'} | last_done={r['last_dones'][-1] if r['last_dones'] else 'NA'}")



RUN torchao_w8a16_action_only_atm_ohb_spatial_task0_4ep_libero_spatial_task0_ep0
SUCCESS returncode= 0 server_crash= False egl_cleanup_warning= True
last_rewards: ['0.00', '0.00', '0.00', '0.00', '0.00', '0.00', '0.00', '0.00', '0.00', '1.00']
last_dones: ['False', 'False', 'False', 'False', 'False', 'False', 'False', 'False', 'False', 'True']
--- tail ---
	err = EGL_NOT_INITIALIZED,
	baseOperation = eglDestroyContext,
	cArguments = (
	),
	result = 0
)
Exception ignored in: <function EGLGLContext.__del__ at 0x7ff0efdb6550>
Traceback (most recent call last):
  File "/content/micromamba-root/envs/libero/lib/python3.8/site-packages/robosuite/renderers/context/egl_context.py", line 155, in __del__
    self.free()
  File "/content/micromamba-root/envs/libero/lib/python3.8/site-packages/robosuite/renderers/context/egl_context.py", line 150, in free
    EGL.eglDestroyContext(EGL_DISPLAY, self._context)
  File "/content/micromamba-root/envs/libero/lib/python3.8/site-packages/OpenGL/error.py",

In [ ]:

# QVLA9. Combine summaries and save CSV/JSON.
import json, pandas as pd
from pathlib import Path

EVAL_RESULTS = RESULTS / "eval_logs"
records = []
for p in sorted(EVAL_RESULTS.glob(f"{TAG}_*.done.json")):
    records.append(json.loads(p.read_text()))

df = pd.DataFrame(records)
if len(df) == 0:
    print("No done summaries yet.")
else:
    display(df)
    df["task_fail"] = (~df["success_detected"].astype(bool)) & (~df["crashed_detected"].astype(bool))
    agg = df.groupby("suite").agg(eval_runs=("log_path", "count"), success=("success_detected", "sum"), task_fail=("task_fail", "sum"), server_crash=("crashed_detected", "sum"), egl_cleanup=("ignored_egl_cleanup_warning", "sum"))
    agg["success_rate"] = agg["success"] / agg["eval_runs"]
    display(agg)
    out_csv = EVAL_RESULTS / f"{TAG}_combined.csv"
    out_json = EVAL_RESULTS / f"{TAG}_combined.json"
    df.to_csv(out_csv, index=False)
    out_json.write_text(json.dumps(records, indent=2))
    print("Saved:", out_csv)
    print("Saved:", out_json)
    print("Scales:", SCALES_PATH)
    print("Calibration buffer:", BUFFER_PATH)


In [ ]:

# QVLA10. Stop server when finished.
# Uncomment when you are done with all eval.
# kill_port_9010()
# subprocess.run("ss -ltnp | grep ':9010' || echo 'port 9010 is free'", shell=True)
